In [22]:
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, END
import os
import json
import time
from dotenv import load_dotenv
from typing import Any, Dict, List, Optional, Literal, TypedDict, Annotated, operator
from dataclasses import dataclass, asdict
import re
from datetime import datetime, timezone
import hashlib


In [23]:
load_dotenv()
GOOGLE_API_KEY = os.getenv("GEMINI_API_KEY")

In [5]:

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite-preview", 
    api_key=GOOGLE_API_KEY,
    temperature=0.1)

llm.invoke("hi").text

'Hello! How can I help you today?'

In [7]:
class DevState(TypedDict):
    raw_input:           str
    intent_manifest:     Optional[Dict]
    compliance_rules:    Optional[Dict]
    ip_clearance:        Optional[Dict]
    architecture:        Optional[Dict]
    generated_code:      Optional[Dict]
    explainability_docs: Optional[Dict]
    security_report:     Optional[Dict]
    quality_report:      Optional[Dict]
    hitl_decisions:      Optional[List]
    audit_log:           Annotated[List[Dict], operator.add]
    drift_alerts:        Optional[List]


In [8]:
def extract_json(text: str) -> dict:
    text = re.sub(r"```(?:json)?", "", text).replace("```", "").strip()
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON object found in LLM response:\n{text}")
    return json.loads(match.group())


def make_audit_entry(agent: str, summary: str, data: dict) -> dict:

    return {
        "agent":     agent,
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "summary":   summary,
        "data":      data,
    }

In [9]:
from datetime import datetime, timezone

timestamp = datetime.now(timezone.utc).isoformat()

print(timestamp)

2026-03-14T12:28:39.272848+00:00


In [10]:
INTENT_SCHEMA = """
{
  "app_type": "string — e.g. REST API, CLI tool, web app",
  "modules": [
    {
      "name": "string",
      "description": "string",
      "tech_stack": ["string"]
    }
  ],
  "constraints": {
    "security": ["string"],
    "compliance": ["string — e.g. GDPR, HIPAA"],
    "performance": ["string"],
    "ip_notes": ["string — any third-party libraries or frameworks mentioned"]
  },
  "acceptance_criteria": ["string"]
}
"""

def intent_agent(state: DevState) -> dict:
    prompt = f"""
You are an AI software architect. Convert the user's description into a
structured intent manifest. Respond ONLY with valid JSON — no explanation,
no markdown, no extra text.

User input:
{state["raw_input"]}

Return exactly this JSON schema (fill in real values, do not include comments):
{INTENT_SCHEMA}
"""

    response = llm.invoke(prompt)
    manifest = extract_json(response.text)

    audit_entry = make_audit_entry(
        agent   = "intent_agent",
        summary = f"Parsed intent for: {state['raw_input'][:80]}",
        data    = {"app_type": manifest.get("app_type"), "modules": [m["name"] for m in manifest.get("modules", [])]}
    )

    return {
        "intent_manifest": manifest,
        "audit_log":       [audit_entry],   
    }


In [56]:
def main():
    state: DevState = {
        "raw_input":           "Build a FastAPI web app with login and dashboard using PostgreSQL",
        "intent_manifest":     None,
        "compliance_rules":    None,
        "ip_clearance":        None,
        "architecture":        None,
        "generated_code":      None,
        "explainability_docs": None,
        "security_report":     None,
        "quality_report":      None,
        "hitl_decisions":      None,
        "audit_log":           [],
        "drift_alerts":        None,
    }

    print("=== Running intent_agent ===")
    result = intent_agent(state)
    state["intent_manifest"] = result["intent_manifest"]
    state["audit_log"] += result["audit_log"]   # simulate operator.add

    print("\nIntent Manifest:")
    print(json.dumps(state["intent_manifest"], indent=2))

    print("\nAudit Log:")
    print(json.dumps(state["audit_log"], indent=2))

if __name__ == "__main__":
    main()

=== Running intent_agent ===

Intent Manifest:
{
  "app_type": "web app",
  "modules": [
    {
      "name": "Authentication Module",
      "description": "User registration and login system with JWT-based session management.",
      "tech_stack": [
        "FastAPI",
        "Passlib",
        "PyJWT"
      ]
    },
    {
      "name": "Dashboard Module",
      "description": "Protected user interface for viewing account data.",
      "tech_stack": [
        "FastAPI",
        "Jinja2",
        "HTML/CSS"
      ]
    },
    {
      "name": "Data Persistence Layer",
      "description": "Relational database management for user credentials and application state.",
      "tech_stack": [
        "PostgreSQL",
        "SQLAlchemy",
        "Alembic"
      ]
    }
  ],
  "constraints": {
    "security": [
      "Password hashing with bcrypt",
      "JWT token expiration",
      "Secure cookie handling"
    ],
    "compliance": [
      "None specified"
    ],
    "performance": [
      "Asyn

In [11]:
import re

KNOWN_LICENSES = {
    # safe for commercial use
    "fastapi":      {"license": "MIT",        "risk": "low"},
    "sqlalchemy":   {"license": "MIT",        "risk": "low"},
    "pydantic":     {"license": "MIT",        "risk": "low"},
    "uvicorn":      {"license": "BSD",        "risk": "low"},
    "postgresql":   {"license": "PostgreSQL", "risk": "low"},
    "alembic":      {"license": "MIT",        "risk": "low"},
    "passlib":      {"license": "BSD",        "risk": "low"},
    "jose":         {"license": "MIT",        "risk": "low"},
    "httpx":        {"license": "BSD",        "risk": "low"},
    "pydantic":     {"license": "MIT",        "risk": "low"},
    "starlette":    {"license": "BSD",        "risk": "low"},
    "asyncpg":      {"license": "Apache-2.0", "risk": "low"},
    # caution — copyleft
    "gpl":          {"license": "GPL",        "risk": "high"},
    "mysql":        {"license": "GPL",        "risk": "high"},
    "celery":       {"license": "BSD",        "risk": "low"},
    "redis":        {"license": "BSD",        "risk": "low"},
}

IP_SCAN_SCHEMA = """
{
  "scanned_libraries": [
    {
      "name": "string",
      "license": "string",
      "risk_level": "low | medium | high",
      "reason": "string — one sentence explanation"
    }
  ],
  "overall_risk": "low | medium | high",
  "flagged_items": ["string — only items with medium or high risk"],
  "recommendation": "string — what to do next"
}
"""

def ip_guard_agent(state: DevState) -> dict:
    manifest = state["intent_manifest"]

    raw_libs = []

    for module in manifest.get("modules", []):
        raw_libs.extend(module.get("tech_stack", []))

    raw_libs.extend(
        manifest.get("constraints", {}).get("ip_notes", [])
    )

    local_flags = []
    for lib in raw_libs:
        key = lib.lower().strip()
        if key in KNOWN_LICENSES and KNOWN_LICENSES[key]["risk"] == "high":
            local_flags.append(f"{lib} ({KNOWN_LICENSES[key]['license']} — copyleft risk)")

    prompt = f"""
You are an IP and software license compliance officer.

Review the following libraries and frameworks extracted from a software project.
For each one, identify its open source license and assess the risk for commercial use.

Libraries to scan:
{json.dumps(raw_libs, indent=2)}

Known high-risk flags detected by local scan (include these in your response):
{json.dumps(local_flags, indent=2) if local_flags else "None detected"}

Risk levels:
- low    : permissive license (MIT, BSD, Apache 2.0, PostgreSQL) — safe for commercial use
- medium : weak copyleft (LGPL, MPL) — usable but requires care
- high   : strong copyleft (GPL, AGPL) — may require open-sourcing your code

Respond ONLY with valid JSON matching this schema exactly, no explanation, no markdown:
{IP_SCAN_SCHEMA}
"""

    response = llm.invoke(prompt)
    clearance = extract_json(response.text)

    overall_risk = clearance.get("overall_risk", "unknown")

    audit_entry = make_audit_entry(
        agent   = "ip_guard_agent",
        summary = f"IP scan complete — overall risk: {overall_risk}",
        data    = {
            "libraries_scanned": len(clearance.get("scanned_libraries", [])),
            "flagged_items":     clearance.get("flagged_items", []),
            "overall_risk":      overall_risk,
        }
    )

    return {
        "ip_clearance": clearance,
        "audit_log":    [audit_entry],
    }

In [58]:
def main():
    state: DevState = {
        "raw_input":           "Build a FastAPI web app with login and dashboard using PostgreSQL",
        "intent_manifest":     None,
        "compliance_rules":    None,
        "ip_clearance":        None,
        "architecture":        None,
        "generated_code":      None,
        "explainability_docs": None,
        "security_report":     None,
        "quality_report":      None,
        "hitl_decisions":      [],
        "audit_log":           [],
        "drift_alerts":        None,
    }

    # agent 1
    print("=== intent_agent ===")
    result = intent_agent(state)
    state["intent_manifest"] = result["intent_manifest"]
    state["audit_log"] += result["audit_log"]
    print(json.dumps(state["intent_manifest"], indent=2))

    # agent 2
    print("\n=== ip_guard_agent ===")
    result = ip_guard_agent(state)
    state["ip_clearance"] = result["ip_clearance"]
    state["audit_log"] += result["audit_log"]
    print(json.dumps(state["ip_clearance"], indent=2))

    # audit trail so far
    print("\n=== Audit Log ===")
    print(json.dumps(state["audit_log"], indent=2))

if __name__ == "__main__":
    main()

=== intent_agent ===
{
  "app_type": "web app",
  "modules": [
    {
      "name": "Authentication Module",
      "description": "User registration and login system with JWT-based session management.",
      "tech_stack": [
        "FastAPI",
        "Passlib",
        "PyJWT"
      ]
    },
    {
      "name": "Dashboard Module",
      "description": "Protected user interface for displaying account-specific data.",
      "tech_stack": [
        "FastAPI",
        "Jinja2",
        "HTML/CSS"
      ]
    },
    {
      "name": "Data Persistence Layer",
      "description": "Relational database management for user credentials and dashboard content.",
      "tech_stack": [
        "PostgreSQL",
        "SQLAlchemy",
        "Alembic"
      ]
    }
  ],
  "constraints": {
    "security": [
      "Password hashing with bcrypt",
      "JWT token expiration",
      "Secure cookie handling"
    ],
    "compliance": [
      "None specified"
    ],
    "performance": [
      "Asynchronous datab

In [12]:

def display_hitl_summary(state: DevState):
    manifest  = state["intent_manifest"]
    clearance = state["ip_clearance"]

    print("\n" + "="*60)
    print("  HITL GATE 1 — INTENT + IP REVIEW")
    print("="*60)

    print("\n[ Intent Manifest ]")
    print(f"  App type : {manifest.get('app_type')}")
    print(f"  Modules  :")
    for m in manifest.get("modules", []):
        print(f"    - {m['name']} → {', '.join(m.get('tech_stack', []))}")
    print(f"  Security constraints  : {manifest.get('constraints', {}).get('security', [])}")
    print(f"  Compliance constraints: {manifest.get('constraints', {}).get('compliance', [])}")
    print(f"  Acceptance criteria   :")
    for ac in manifest.get("acceptance_criteria", []):
        print(f"    - {ac}")

    print("\n[ IP Clearance Report ]")
    risk = clearance.get("overall_risk", "unknown")
    risk_display = {"low": "LOW", "medium": "MEDIUM ⚠", "high": "HIGH ✗"}.get(risk, risk.upper())
    print(f"  Overall risk : {risk_display}")
    
    flagged = clearance.get("flagged_items", [])
    if flagged:
        print(f"  Flagged items:")
        for item in flagged:
            print(f"    ! {item}")
    else:
        print(f"  Flagged items: none")

    print(f"  Recommendation: {clearance.get('recommendation', 'N/A')}")

    print("\n[ Scanned Libraries ]")
    for lib in clearance.get("scanned_libraries", []):
        risk_tag = {"low": "[ok]", "medium": "[warn]", "high": "[RISK]"}.get(lib["risk_level"], "[?]")
        print(f"  {risk_tag:8s} {lib['name']:20s} {lib['license']:15s} — {lib['reason']}")

    print("\n" + "-"*60)


def get_human_decision(state: DevState) -> dict:

    display_hitl_summary(state)

    print("\nOptions:")
    print("  [A] Approve — proceed to compliance + architecture agents")
    print("  [R] Reject  — send back to intent_agent with feedback")
    print("  [M] Modify  — approve with notes (human adds constraints)")
    print()

    while True:
        choice = input("Your decision [A/R/M]: ").strip().upper()
        if choice in ("A", "R", "M"):
            break
        print("  Invalid input — please enter A, R, or M")

    feedback    = None
    extra_notes = None

    if choice == "R":
        feedback = input("Rejection reason (will be passed back to intent_agent): ").strip()

    if choice == "M":
        extra_notes = input("Additional constraints or notes to add: ").strip()

    decision = {
        "gate":        "hitl_gate_1",
        "approved":    choice in ("A", "M"),
        "choice":      choice,
        "approver":    input("Your name (for audit log): ").strip(),
        "feedback":    feedback,
        "extra_notes": extra_notes,
        "timestamp":   datetime.now(timezone.utc).isoformat(),
    }

    return decision


def hitl_gate_1(state: DevState) -> dict:

    if not state.get("intent_manifest") or not state.get("ip_clearance"):
        raise ValueError("hitl_gate_1 requires both intent_manifest and ip_clearance in state")

    decision = get_human_decision(state)

    updated_manifest = state["intent_manifest"]

    if decision.get("extra_notes"):
        existing = updated_manifest.get("constraints", {})
        existing.setdefault("human_notes", []).append(decision["extra_notes"])
        updated_manifest["constraints"] = existing

    # if rejected, stamp the feedback onto raw_input so intent_agent can use it
    updated_raw = state["raw_input"]
    if not decision["approved"] and decision.get("feedback"):
        updated_raw = f"{state['raw_input']}\n\n[HITL feedback]: {decision['feedback']}"

    audit_entry = make_audit_entry(
        agent   = "hitl_gate_1",
        summary = f"Gate 1 decision: {decision['choice']} by {decision['approver']}",
        data    = decision,
    )

    return {
        "hitl_decisions":  (state["hitl_decisions"] or []) + [decision],
        "intent_manifest": updated_manifest,
        "raw_input":       updated_raw,      
        "audit_log":       [audit_entry],
    }


def route_after_hitl_1(state: DevState) -> str:

    last_decision = state["hitl_decisions"][-1]

    if last_decision["approved"]:
        return "compliance_agent" 
    else:
        return "intent_agent"       

In [ ]:
def main():
    state: DevState = {
        "raw_input":           "Build a FastAPI web app with login and dashboard using PostgreSQL",
        "intent_manifest":     None,
        "compliance_rules":    None,
        "ip_clearance":        None,
        "architecture":        None,
        "generated_code":      None,
        "explainability_docs": None,
        "security_report":     None,
        "quality_report":      None,
        "hitl_decisions":      [],
        "audit_log":           [],
        "drift_alerts":        None,
    }

    print("=== intent_agent ===")
    result = intent_agent(state)
    state["intent_manifest"] = result["intent_manifest"]
    state["audit_log"] += result["audit_log"]

    print("\n=== ip_guard_agent ===")
    result = ip_guard_agent(state)
    state["ip_clearance"] = result["ip_clearance"]
    state["audit_log"] += result["audit_log"]

    print("\n=== hitl_gate_1 ===")
    result = hitl_gate_1(state)
    state["hitl_decisions"]  = result["hitl_decisions"]
    state["intent_manifest"] = result["intent_manifest"]
    state["raw_input"]       = result["raw_input"]
    state["audit_log"]      += result["audit_log"]

    next_node = route_after_hitl_1(state)
    print(f"\n>> Routing to: {next_node}")

    if next_node == "intent_agent":
        print(">> Looping back — re-running intent_agent with feedback...")
        result = intent_agent(state)   
        state["intent_manifest"] = result["intent_manifest"]
        state["audit_log"] += result["audit_log"]

    print("\n=== Full Audit Log ===")
    for entry in state["audit_log"]:
        print(f"  [{entry['timestamp']}] {entry['agent']:20s} — {entry['summary']}")

if __name__ == "__main__":
    main()

=== intent_agent ===

=== ip_guard_agent ===

=== hitl_gate_1 ===

  HITL GATE 1 — INTENT + IP REVIEW

[ Intent Manifest ]
  App type : web app
  Modules  :
    - Authentication Service → FastAPI, Passlib, PyJWT
    - Dashboard Module → FastAPI, Jinja2, HTML5
    - Data Persistence Layer → PostgreSQL, SQLAlchemy, Alembic
  Security constraints  : ['Password hashing with Argon2 or BCrypt', 'JWT token expiration', 'HTTPS enforcement']
  Compliance constraints: ['None specified']
  Acceptance criteria   :
    - User can register a new account
    - User can log in and receive a session token
    - Dashboard is inaccessible to unauthenticated users
    - Data is successfully persisted in PostgreSQL

[ IP Clearance Report ]
  Overall risk : LOW
  Flagged items: none
  Recommendation: All identified libraries are under permissive licenses; no further legal action is required for commercial deployment.

[ Scanned Libraries ]
  [ok]     FastAPI              MIT             — Permissive license

In [13]:
COMPLIANCE_FRAMEWORKS = {
    "gdpr": {
        "full_name": "General Data Protection Regulation",
        "triggers":  ["user data", "login", "personal data", "email", "profile", "authentication"],
        "rules": [
            "User data must be encrypted at rest and in transit",
            "Users must be able to request data deletion",
            "Explicit consent required before data collection",
            "Data breach notification within 72 hours",
            "Minimal data collection principle must be followed",
        ]
    },
    "owasp": {
        "full_name": "OWASP Top 10",
        "triggers":  ["login", "authentication", "api", "web", "dashboard", "jwt", "password"],
        "rules": [
            "Protect against SQL injection via parameterised queries",
            "Implement proper session management and token expiry",
            "Enforce HTTPS and secure cookie flags",
            "Rate limit authentication endpoints",
            "Validate and sanitise all user inputs",
        ]
    },
    "hipaa": {
        "full_name": "Health Insurance Portability and Accountability Act",
        "triggers":  ["health", "medical", "patient", "diagnosis", "prescription", "ehr"],
        "rules": [
            "PHI must be encrypted at rest and in transit",
            "Access logs must be maintained for all PHI access",
            "Role-based access control for all health data",
            "Business Associate Agreements required for third parties",
        ]
    },
    "pci_dss": {
        "full_name": "Payment Card Industry Data Security Standard",
        "triggers":  ["payment", "card", "billing", "checkout", "stripe", "transaction"],
        "rules": [
            "Card data must never be stored in plaintext",
            "Use tokenisation for all payment data",
            "Quarterly vulnerability scans required",
            "Strict access control to cardholder data",
        ]
    },
}

COMPLIANCE_SCHEMA = """
{
  "applicable_frameworks": [
    {
      "name": "string — e.g. GDPR, OWASP Top 10",
      "reason": "string — why this framework applies",
      "rules": ["string — specific rule that must be implemented"],
      "priority": "mandatory | recommended"
    }
  ],
  "consolidated_rules": [
    {
      "rule": "string — the actual requirement",
      "framework": "string — which framework it comes from",
      "implementation_hint": "string — how to implement in the tech stack"
    }
  ],
  "gaps": ["string — things in the intent manifest that may violate compliance"],
  "overall_compliance_risk": "low | medium | high"
}
"""

def detect_frameworks_locally(manifest: dict) -> list[str]:

    manifest_text = json.dumps(manifest).lower()

    triggered = []
    for fw_key, fw in COMPLIANCE_FRAMEWORKS.items():
        if any(trigger in manifest_text for trigger in fw["triggers"]):
            triggered.append(fw_key)

    return triggered


def compliance_agent(state: DevState) -> dict:
    manifest         = state["intent_manifest"]
    ip_clearance     = state["ip_clearance"]
    triggered_fws    = detect_frameworks_locally(manifest)

    framework_context = ""
    for fw_key in triggered_fws:
        fw = COMPLIANCE_FRAMEWORKS[fw_key]
        framework_context += f"\n{fw['full_name']} ({fw_key.upper()}):\n"
        for rule in fw["rules"]:
            framework_context += f"  - {rule}\n"

    prompt = f"""
You are a regulatory compliance officer reviewing a software project.

Intent Manifest:
{json.dumps(manifest, indent=2)}

IP Clearance Summary:
- Overall risk: {ip_clearance.get('overall_risk')}
- Flagged items: {ip_clearance.get('flagged_items', [])}

Locally detected applicable frameworks and their rules:
{framework_context if framework_context else "None auto-detected — use your judgment."}

Your tasks:
1. Confirm which compliance frameworks apply and why
2. List every specific rule that must be implemented given the tech stack
3. Identify any gaps or risks in the current intent manifest
4. Give each rule an implementation hint specific to the tech stack

Respond ONLY with valid JSON matching this schema, no explanation, no markdown:
{COMPLIANCE_SCHEMA}
"""

    response  = llm.invoke(prompt)
    rules     = extract_json(response.text)

    risk      = rules.get("overall_compliance_risk", "unknown")
    fw_names  = [f["name"] for f in rules.get("applicable_frameworks", [])]
    gaps      = rules.get("gaps", [])

    audit_entry = make_audit_entry(
        agent   = "compliance_agent",
        summary = f"Compliance mapped — risk: {risk} | frameworks: {', '.join(fw_names)}",
        data    = {
            "frameworks_detected":   triggered_fws,
            "frameworks_confirmed":  fw_names,
            "consolidated_rules":    len(rules.get("consolidated_rules", [])),
            "gaps_found":            len(gaps),
            "overall_risk":          risk,
        }
    )

    return {
        "compliance_rules": rules,
        "audit_log":        [audit_entry],
    }

In [67]:
def main():
    state: DevState = {
        "raw_input":           "Build a FastAPI web app with login and dashboard using PostgreSQL",
        "intent_manifest":     None,
        "compliance_rules":    None,
        "ip_clearance":        None,
        "architecture":        None,
        "generated_code":      None,
        "explainability_docs": None,
        "security_report":     None,
        "quality_report":      None,
        "hitl_decisions":      [],
        "audit_log":           [],
        "drift_alerts":        None,
    }

    # ── agent 1 ──
    print("=== intent_agent ===")
    result = intent_agent(state)
    state["intent_manifest"] = result["intent_manifest"]
    state["audit_log"] += result["audit_log"]

    # ── agent 2 ──
    print("\n=== ip_guard_agent ===")
    result = ip_guard_agent(state)
    state["ip_clearance"] = result["ip_clearance"]
    state["audit_log"] += result["audit_log"]

    # ── HITL gate 1 — loop until approved ──────────────────────────
    max_iterations = 3
    iteration      = 0
    approved       = False          # ← THIS was missing in your version

    while not approved and iteration < max_iterations:
        iteration += 1
        print(f"\n=== hitl_gate_1 (attempt {iteration}) ===")

        result = hitl_gate_1(state)
        state["hitl_decisions"]  = result["hitl_decisions"]
        state["intent_manifest"] = result["intent_manifest"]
        state["raw_input"]       = result["raw_input"]
        state["audit_log"]      += result["audit_log"]

        last = state["hitl_decisions"][-1]

        if last["choice"] == "R":
            print(f"\n>> Rejected — reason: {last['feedback']}")
            print(">> Re-running intent_agent with feedback...")
            result = intent_agent(state)
            state["intent_manifest"] = result["intent_manifest"]
            state["audit_log"] += result["audit_log"]

            print(">> Re-running ip_guard_agent on updated manifest...")
            result = ip_guard_agent(state)
            state["ip_clearance"] = result["ip_clearance"]   # ← ip fix from earlier too
            state["audit_log"] += result["audit_log"]

            print("\n>> Updated manifest modules:")
            for m in state["intent_manifest"].get("modules", []):
                print(f"   - {m['name']} → {', '.join(m.get('tech_stack', []))}")

        elif last["choice"] == "M" and last.get("extra_notes"):
            print(f"\n>> Modified — applying: {last['extra_notes']}")
            state["raw_input"] = (
                f"{state['raw_input']}\n\n"
                f"[Human modification]: {last['extra_notes']}\n"
                f"Please update the intent manifest to reflect this change exactly."
            )
            result = intent_agent(state)
            state["intent_manifest"] = result["intent_manifest"]
            state["audit_log"] += result["audit_log"]

            result = ip_guard_agent(state)
            state["ip_clearance"] = result["ip_clearance"]
            state["audit_log"] += result["audit_log"]

            print("\n>> Updated manifest modules:")
            for m in state["intent_manifest"].get("modules", []):
                print(f"   - {m['name']} → {', '.join(m.get('tech_stack', []))}")

            approved = True

        else:  # A
            approved = True
            print("\n>> Approved — proceeding to compliance_agent")

    if not approved:
        print(f"\n>> Max iterations ({max_iterations}) reached — escalating")
        return  # ← stop pipeline, don't proceed

    # ── compliance_agent only runs if approved ──────────────────────
    if approved:
        print("\n=== compliance_agent ===")
        result = compliance_agent(state)
        state["compliance_rules"] = result["compliance_rules"]
        state["audit_log"] += result["audit_log"]

        print("\n[ Applicable Frameworks ]")
        for fw in state["compliance_rules"].get("applicable_frameworks", []):
            print(f"  {fw['priority'].upper():12s} {fw['name']}")
            print(f"               {fw['reason']}")

        print("\n[ Consolidated Rules ]")
        for i, r in enumerate(state["compliance_rules"].get("consolidated_rules", []), 1):
            print(f"  {i:2}. [{r['framework']}] {r['rule']}")
            print(f"      Hint: {r['implementation_hint']}")

        print("\n[ Gaps Detected ]")
        gaps = state["compliance_rules"].get("gaps", [])
        if gaps:
            for g in gaps:
                print(f"  ! {g}")
        else:
            print("  None detected")

        print(f"\n[ Overall Compliance Risk ] "
              f"{state['compliance_rules'].get('overall_compliance_risk', '').upper()}")

    # ── full audit trail ────────────────────────────────────────────
    print("\n=== Full Audit Log ===")
    for entry in state["audit_log"]:
        print(f"  [{entry['timestamp']}] {entry['agent']:20s} — {entry['summary']}")


if __name__ == "__main__":
    main()

=== intent_agent ===

=== ip_guard_agent ===

=== hitl_gate_1 (attempt 1) ===

  HITL GATE 1 — INTENT + IP REVIEW

[ Intent Manifest ]
  App type : web app
  Modules  :
    - Authentication Service → FastAPI, Passlib, PyJWT
    - Dashboard Module → FastAPI, Jinja2, HTML/CSS
    - Data Persistence Layer → PostgreSQL, SQLAlchemy, Alembic
  Security constraints  : ['Password hashing with Argon2 or BCrypt', 'JWT token expiration', 'CORS policy configuration']
  Compliance constraints: ['None specified']
  Acceptance criteria   :
    - User can register a new account
    - User can log in and receive a session token
    - Dashboard is inaccessible to unauthenticated users
    - Data is successfully persisted in PostgreSQL

[ IP Clearance Report ]
  Overall risk : LOW
  Flagged items: none
  Recommendation: All identified libraries are under permissive licenses; no further legal action is required for commercial deployment.

[ Scanned Libraries ]
  [ok]     FastAPI              MIT          

In [14]:
ARCHITECTURE_PATTERNS = {
    "layered": {
        "description": "Traditional N-tier: presentation, business logic, data",
        "best_for":    ["web apps", "dashboards", "CRUD applications"],
        "trade_offs":  {"scalability": "medium", "complexity": "low", "security": "high"}
    },
    "microservices": {
        "description": "Independent services communicating via APIs",
        "best_for":    ["large teams", "high scale", "independent deployments"],
        "trade_offs":  {"scalability": "high", "complexity": "high", "security": "medium"}
    },
    "modular_monolith": {
        "description": "Single deployable unit with strict internal module boundaries",
        "best_for":    ["small teams", "early stage", "web apps"],
        "trade_offs":  {"scalability": "medium", "complexity": "low", "security": "high"}
    },
}

ARCHITECTURE_SCHEMA = """
{
  "selected_pattern": "string — layered | microservices | modular_monolith",
  "pattern_rationale": "string — why this pattern was chosen",
  "layers": [
    {
      "name": "string — e.g. API Layer, Auth Layer, Data Layer",
      "responsibility": "string — what this layer does",
      "components": ["string — specific classes, services or modules"],
      "tech": ["string — specific libraries or tools"],
      "compliance_controls": ["string — which compliance rules this layer satisfies"]
    }
  ],
  "infrastructure": {
    "database":    "string",
    "cache":       "string",
    "tls":         "string",
    "rate_limiter":"string",
    "audit_store": "string"
  },
  "security_controls": ["string — specific security measures built into the architecture"],
  "trade_off_matrix": {
    "scalability":  "low | medium | high",
    "complexity":   "low | medium | high",
    "security":     "low | medium | high",
    "compliance_fit":"low | medium | high"
  },
  "gaps_addressed": ["string — which compliance gaps from previous agent are now covered"],
  "residual_risks": ["string — anything still not addressed"]
}
"""

def architecture_agent(state: DevState) -> dict:
    manifest         = state["intent_manifest"]
    compliance       = state["compliance_rules"]
    ip_clearance     = state["ip_clearance"]

    # pull compliance gaps so arch agent explicitly addresses them
    gaps             = compliance.get("gaps", [])
    consolidated     = compliance.get("consolidated_rules", [])
    frameworks       = [f["name"] for f in compliance.get("applicable_frameworks", [])]

    # build pattern context for the prompt
    pattern_context  = ""
    for name, pattern in ARCHITECTURE_PATTERNS.items():
        pattern_context += f"\n{name}:\n"
        pattern_context += f"  Description : {pattern['description']}\n"
        pattern_context += f"  Best for    : {', '.join(pattern['best_for'])}\n"
        pattern_context += f"  Trade-offs  : {pattern['trade_offs']}\n"

    prompt = f"""
You are a senior software architect designing a production-grade system.

Intent Manifest:
{json.dumps(manifest, indent=2)}

Compliance Frameworks Active: {', '.join(frameworks)}

Compliance Rules to Satisfy:
{json.dumps(consolidated, indent=2)}

Compliance Gaps That Must Be Addressed in Architecture:
{json.dumps(gaps, indent=2)}

IP Clearance:
- Overall risk  : {ip_clearance.get('overall_risk')}
- Flagged items : {ip_clearance.get('flagged_items', [])}

Available Architecture Patterns:
{pattern_context}

Your tasks:
1. Select the most appropriate architecture pattern given the app type,
   team size implied by the manifest, and compliance requirements
2. Define each layer with its responsibilities, components, and tech
3. Map every compliance rule to the layer that satisfies it
4. Address every compliance gap explicitly in either a layer or infrastructure
5. Score the architecture on the trade-off matrix
6. List any residual risks that could not be fully addressed

Respond ONLY with valid JSON matching this schema, no explanation, no markdown:
{ARCHITECTURE_SCHEMA}
"""

    response = llm.invoke(prompt)
    arch     = extract_json(response.text)

    pattern  = arch.get("selected_pattern", "unknown")
    gaps_addressed = arch.get("gaps_addressed", [])
    residual = arch.get("residual_risks", [])

    audit_entry = make_audit_entry(
        agent   = "architecture_agent",
        summary = f"Architecture synthesised — pattern: {pattern} | gaps addressed: {len(gaps_addressed)}",
        data    = {
            "pattern":          pattern,
            "layers":           [l["name"] for l in arch.get("layers", [])],
            "gaps_addressed":   gaps_addressed,
            "residual_risks":   residual,
            "trade_off_matrix": arch.get("trade_off_matrix", {}),
        }
    )

    return {
        "architecture": arch,
        "audit_log":    [audit_entry],
    }

In [69]:
def main():
    state: DevState = {
        "raw_input":           "Build a FastAPI web app with login and dashboard using PostgreSQL",
        "intent_manifest":     None,
        "compliance_rules":    None,
        "ip_clearance":        None,
        "architecture":        None,
        "generated_code":      None,
        "explainability_docs": None,
        "security_report":     None,
        "quality_report":      None,
        "hitl_decisions":      [],
        "audit_log":           [],
        "drift_alerts":        None,
    }

    # ── agent 1 ──
    print("=== intent_agent ===")
    result = intent_agent(state)
    state["intent_manifest"] = result["intent_manifest"]
    state["audit_log"] += result["audit_log"]

    # ── agent 2 ──
    print("\n=== ip_guard_agent ===")
    result = ip_guard_agent(state)
    state["ip_clearance"] = result["ip_clearance"]
    state["audit_log"] += result["audit_log"]

    # ── HITL gate 1 — loop until approved ──────────────────────────
    max_iterations = 3
    iteration      = 0
    approved       = False          # ← THIS was missing in your version

    while not approved and iteration < max_iterations:
        iteration += 1
        print(f"\n=== hitl_gate_1 (attempt {iteration}) ===")

        result = hitl_gate_1(state)
        state["hitl_decisions"]  = result["hitl_decisions"]
        state["intent_manifest"] = result["intent_manifest"]
        state["raw_input"]       = result["raw_input"]
        state["audit_log"]      += result["audit_log"]

        last = state["hitl_decisions"][-1]

        if last["choice"] == "R":
            print(f"\n>> Rejected — reason: {last['feedback']}")
            print(">> Re-running intent_agent with feedback...")
            result = intent_agent(state)
            state["intent_manifest"] = result["intent_manifest"]
            state["audit_log"] += result["audit_log"]

            print(">> Re-running ip_guard_agent on updated manifest...")
            result = ip_guard_agent(state)
            state["ip_clearance"] = result["ip_clearance"]   # ← ip fix from earlier too
            state["audit_log"] += result["audit_log"]

            print("\n>> Updated manifest modules:")
            for m in state["intent_manifest"].get("modules", []):
                print(f"   - {m['name']} → {', '.join(m.get('tech_stack', []))}")

        elif last["choice"] == "M" and last.get("extra_notes"):
            print(f"\n>> Modified — applying: {last['extra_notes']}")
            state["raw_input"] = (
                f"{state['raw_input']}\n\n"
                f"[Human modification]: {last['extra_notes']}\n"
                f"Please update the intent manifest to reflect this change exactly."
            )
            result = intent_agent(state)
            state["intent_manifest"] = result["intent_manifest"]
            state["audit_log"] += result["audit_log"]

            result = ip_guard_agent(state)
            state["ip_clearance"] = result["ip_clearance"]
            state["audit_log"] += result["audit_log"]

            print("\n>> Updated manifest modules:")
            for m in state["intent_manifest"].get("modules", []):
                print(f"   - {m['name']} → {', '.join(m.get('tech_stack', []))}")

            approved = True

        else:  # A
            approved = True
            print("\n>> Approved — proceeding to compliance_agent")

    if not approved:
        print(f"\n>> Max iterations ({max_iterations}) reached — escalating")
        return  # ← stop pipeline, don't proceed

    # ── compliance_agent only runs if approved ──────────────────────
    if approved:
        print("\n=== compliance_agent ===")
        result = compliance_agent(state)
        state["compliance_rules"] = result["compliance_rules"]
        state["audit_log"] += result["audit_log"]

        print("\n[ Applicable Frameworks ]")
        for fw in state["compliance_rules"].get("applicable_frameworks", []):
            print(f"  {fw['priority'].upper():12s} {fw['name']}")
            print(f"               {fw['reason']}")

        print("\n[ Consolidated Rules ]")
        for i, r in enumerate(state["compliance_rules"].get("consolidated_rules", []), 1):
            print(f"  {i:2}. [{r['framework']}] {r['rule']}")
            print(f"      Hint: {r['implementation_hint']}")

        print("\n[ Gaps Detected ]")
        gaps = state["compliance_rules"].get("gaps", [])
        if gaps:
            for g in gaps:
                print(f"  ! {g}")
        else:
            print("  None detected")

        print(f"\n[ Overall Compliance Risk ] "
              f"{state['compliance_rules'].get('overall_compliance_risk', '').upper()}")

    # ── architecture_agent ──────────────────────────────────────────
    print("\n=== architecture_agent ===")
    result = architecture_agent(state)
    state["architecture"] = result["architecture"]
    state["audit_log"] += result["audit_log"]

    arch = state["architecture"]

    print(f"\n[ Selected Pattern ]  {arch.get('selected_pattern').upper()}")
    print(f"  Rationale: {arch.get('pattern_rationale')}")

    print("\n[ Trade-off Matrix ]")
    for k, v in arch.get("trade_off_matrix", {}).items():
        bar = {"low": "░░░░░░", "medium": "███░░░", "high": "██████"}.get(v, "?")
        print(f"  {k:20s} {bar}  {v.upper()}")

    print("\n[ Layers ]")
    for layer in arch.get("layers", []):
        print(f"\n  {layer['name']}")
        print(f"    Responsibility : {layer['responsibility']}")
        print(f"    Tech           : {', '.join(layer.get('tech', []))}")
        print(f"    Components     : {', '.join(layer.get('components', []))}")
        if layer.get("compliance_controls"):
            print(f"    Satisfies      : {', '.join(layer.get('compliance_controls', []))}")

    print("\n[ Infrastructure ]")
    for k, v in arch.get("infrastructure", {}).items():
        print(f"  {k:15s} : {v}")

    print("\n[ Security Controls ]")
    for sc in arch.get("security_controls", []):
        print(f"  + {sc}")

    print("\n[ Compliance Gaps Addressed ]")
    for g in arch.get("gaps_addressed", []):
        print(f"  [resolved] {g}")

    if arch.get("residual_risks"):
        print("\n[ Residual Risks ]")
        for r in arch.get("residual_risks", []):
            print(f"  [!] {r}")

    print("\n=== Full Audit Log ===")
    for entry in state["audit_log"]:
        print(f"  [{entry['timestamp']}] {entry['agent']:20s} — {entry['summary']}")


if __name__ == "__main__":
    main()

=== intent_agent ===

=== ip_guard_agent ===

=== hitl_gate_1 (attempt 1) ===

  HITL GATE 1 — INTENT + IP REVIEW

[ Intent Manifest ]
  App type : web app
  Modules  :
    - Authentication Module → FastAPI, Passlib, PyJWT
    - Dashboard Module → FastAPI, Jinja2, HTML/CSS
    - Data Persistence Layer → PostgreSQL, SQLAlchemy, Alembic
  Security constraints  : ['Password hashing with Bcrypt', 'JWT token expiration', 'CORS policy configuration']
  Compliance constraints: ['None']
  Acceptance criteria   :
    - User can register a new account
    - User can log in and receive a session token
    - Dashboard is inaccessible to unauthenticated users
    - Data is successfully persisted in PostgreSQL

[ IP Clearance Report ]
  Overall risk : LOW
  Flagged items: none
  Recommendation: All identified libraries are under permissive licenses; no further action is required for compliance.

[ Scanned Libraries ]
  [ok]     FastAPI              MIT             — Permissive license allowing comme

In [15]:
def display_hitl2_summary(state: DevState):
    arch       = state["architecture"]
    compliance = state["compliance_rules"]

    print("\n" + "="*60)
    print("  HITL GATE 2 — ARCHITECTURE + COMPLIANCE REVIEW")
    print("="*60)

    print(f"\n[ Selected Pattern ]  {arch.get('selected_pattern', '').upper()}")
    print(f"  Rationale: {arch.get('pattern_rationale')}")

    print("\n[ Trade-off Matrix ]")
    for k, v in arch.get("trade_off_matrix", {}).items():
        bar = {"low": "░░░░░░", "medium": "███░░░", "high": "██████"}.get(v, "?")
        print(f"  {k:20s} {bar}  {v.upper()}")

    print("\n[ Layers ]")
    for layer in arch.get("layers", []):
        print(f"  {layer['name']}")
        print(f"    Tech      : {', '.join(layer.get('tech', []))}")
        print(f"    Satisfies : {', '.join(layer.get('compliance_controls', []))}")

    print("\n[ Infrastructure ]")
    for k, v in arch.get("infrastructure", {}).items():
        print(f"  {k:15s} : {v}")

    print("\n[ Compliance Gaps Addressed ]")
    original_gaps = compliance.get("gaps", [])
    gaps_addressed = arch.get("gaps_addressed", [])
    
    # cross check — flag any gap not addressed
    unresolved = []
    for gap in original_gaps:
        matched = any(
            any(word in addressed.lower() for word in gap.lower().split()[:3])
            for addressed in gaps_addressed
        )
        if matched:
            print(f"  [ok] {gap}")
        else:
            print(f"  [!!] UNRESOLVED: {gap}")
            unresolved.append(gap)

    if arch.get("residual_risks"):
        print("\n[ Residual Risks — Human Sign-off Required ]")
        for r in arch.get("residual_risks", []):
            print(f"  [!] {r}")

    if unresolved:
        print("\n[ WARNING — Unresolved Gaps Detected ]")
        for u in unresolved:
            print(f"  [!!] {u}")

    print("\n" + "-"*60)


def get_human_decision_2(state: DevState) -> dict:
    display_hitl2_summary(state)

    arch     = state["architecture"]
    residual = arch.get("residual_risks", [])

    # auto warn if residual risks exist
    if residual:
        print(f"\n  NOTE: {len(residual)} residual risk(s) require acknowledgement:")
        for r in residual:
            print(f"    [!] {r}")
        print()

    print("Options:")
    print("  [A] Approve — proceed to codegen + explainability agents")
    print("  [R] Reject  — send back to architecture_agent with feedback")
    print("  [M] Modify  — approve with additional arch constraints")
    print()

    while True:
        choice = input("Your decision [A/R/M]: ").strip().upper()
        if choice in ("A", "R", "M"):
            break
        print("  Invalid input — please enter A, R, or M")

    feedback    = None
    extra_notes = None
    risk_acknowledged = False

    if choice == "R":
        feedback = input("Rejection reason (passed back to architecture_agent): ").strip()

    if choice == "M":
        extra_notes = input("Additional architecture constraints: ").strip()

    # force acknowledgement if residual risks exist
    if residual and choice in ("A", "M"):
        print("\n  Residual risks must be explicitly acknowledged before approval.")
        ack = input("  Type ACKNOWLEDGE to confirm you accept these risks: ").strip().upper()
        risk_acknowledged = ack == "ACKNOWLEDGE"
        if not risk_acknowledged:
            print("  Acknowledgement not confirmed — defaulting to rejection.")
            choice   = "R"
            feedback = "Residual risks not acknowledged by reviewer."

    decision = {
        "gate":               "hitl_gate_2",
        "approved":           choice in ("A", "M"),
        "choice":             choice,
        "approver":           input("Your name (for audit log): ").strip(),
        "feedback":           feedback,
        "extra_notes":        extra_notes,
        "risk_acknowledged":  risk_acknowledged,
        "residual_risks_seen": arch.get("residual_risks", []),
        "timestamp":          datetime.now(timezone.utc).isoformat(),
    }

    return decision


def hitl_gate_2(state: DevState) -> dict:

    if not state.get("architecture") or not state.get("compliance_rules"):
        raise ValueError("hitl_gate_2 requires architecture and compliance_rules in state")

    decision = get_human_decision_2(state)

    updated_arch = state["architecture"]

    # merge extra constraints into architecture if modified
    if decision.get("extra_notes"):
        updated_arch.setdefault("human_constraints", []).append(decision["extra_notes"])

    # if rejected stamp feedback for architecture_agent rerun
    updated_raw = state["raw_input"]
    if not decision["approved"] and decision.get("feedback"):
        updated_raw = (
            f"{state['raw_input']}\n\n"
            f"[HITL Gate 2 feedback]: {decision['feedback']}"
        )

    audit_entry = make_audit_entry(
        agent   = "hitl_gate_2",
        summary = f"Gate 2 decision: {decision['choice']} by {decision['approver']} | risks acknowledged: {decision['risk_acknowledged']}",
        data    = decision,
    )

    return {
        "hitl_decisions": (state["hitl_decisions"] or []) + [decision],
        "architecture":   updated_arch,
        "raw_input":      updated_raw,
        "audit_log":      [audit_entry],
    }


def route_after_hitl_2(state: DevState) -> str:
    last = state["hitl_decisions"][-1]

    if last["gate"] != "hitl_gate_2":
        raise ValueError("route_after_hitl_2 called but last decision is not from gate 2")

    if last["approved"]:
        return "codegen_agent"
    else:
        return "architecture_agent"

In [13]:
def main():
    state: DevState = {
        "raw_input":           "Build a FastAPI web app with login and dashboard using PostgreSQL",
        "intent_manifest":     None,
        "compliance_rules":    None,
        "ip_clearance":        None,
        "architecture":        None,
        "generated_code":      None,
        "explainability_docs": None,
        "security_report":     None,
        "quality_report":      None,
        "hitl_decisions":      [],
        "audit_log":           [],
        "drift_alerts":        None,
    }

    # ── agent 1 ──
    print("=== intent_agent ===")
    result = intent_agent(state)
    state["intent_manifest"] = result["intent_manifest"]
    state["audit_log"] += result["audit_log"]

    # ── agent 2 ──
    print("\n=== ip_guard_agent ===")
    result = ip_guard_agent(state)
    state["ip_clearance"] = result["ip_clearance"]
    state["audit_log"] += result["audit_log"]

    # ── HITL gate 1 — loop until approved ──────────────────────────
    max_iterations = 3
    iteration      = 0
    approved       = False          # ← THIS was missing in your version

    while not approved and iteration < max_iterations:
        iteration += 1
        print(f"\n=== hitl_gate_1 (attempt {iteration}) ===")

        result = hitl_gate_1(state)
        state["hitl_decisions"]  = result["hitl_decisions"]
        state["intent_manifest"] = result["intent_manifest"]
        state["raw_input"]       = result["raw_input"]
        state["audit_log"]      += result["audit_log"]

        last = state["hitl_decisions"][-1]

        if last["choice"] == "R":
            print(f"\n>> Rejected — reason: {last['feedback']}")
            print(">> Re-running intent_agent with feedback...")
            result = intent_agent(state)
            state["intent_manifest"] = result["intent_manifest"]
            state["audit_log"] += result["audit_log"]

            print(">> Re-running ip_guard_agent on updated manifest...")
            result = ip_guard_agent(state)
            state["ip_clearance"] = result["ip_clearance"]   # ← ip fix from earlier too
            state["audit_log"] += result["audit_log"]

            print("\n>> Updated manifest modules:")
            for m in state["intent_manifest"].get("modules", []):
                print(f"   - {m['name']} → {', '.join(m.get('tech_stack', []))}")

        elif last["choice"] == "M" and last.get("extra_notes"):
            print(f"\n>> Modified — applying: {last['extra_notes']}")
            state["raw_input"] = (
                f"{state['raw_input']}\n\n"
                f"[Human modification]: {last['extra_notes']}\n"
                f"Please update the intent manifest to reflect this change exactly."
            )
            result = intent_agent(state)
            state["intent_manifest"] = result["intent_manifest"]
            state["audit_log"] += result["audit_log"]

            result = ip_guard_agent(state)
            state["ip_clearance"] = result["ip_clearance"]
            state["audit_log"] += result["audit_log"]

            print("\n>> Updated manifest modules:")
            for m in state["intent_manifest"].get("modules", []):
                print(f"   - {m['name']} → {', '.join(m.get('tech_stack', []))}")

            approved = True

        else:  # A
            approved = True
            print("\n>> Approved — proceeding to compliance_agent")

    if not approved:
        print(f"\n>> Max iterations ({max_iterations}) reached — escalating")
        return  # ← stop pipeline, don't proceed

    # ── compliance_agent only runs if approved ──────────────────────
    if approved:
        print("\n=== compliance_agent ===")
        result = compliance_agent(state)
        state["compliance_rules"] = result["compliance_rules"]
        state["audit_log"] += result["audit_log"]

        print("\n[ Applicable Frameworks ]")
        for fw in state["compliance_rules"].get("applicable_frameworks", []):
            print(f"  {fw['priority'].upper():12s} {fw['name']}")
            print(f"               {fw['reason']}")

        print("\n[ Consolidated Rules ]")
        for i, r in enumerate(state["compliance_rules"].get("consolidated_rules", []), 1):
            print(f"  {i:2}. [{r['framework']}] {r['rule']}")
            print(f"      Hint: {r['implementation_hint']}")

        print("\n[ Gaps Detected ]")
        gaps = state["compliance_rules"].get("gaps", [])
        if gaps:
            for g in gaps:
                print(f"  ! {g}")
        else:
            print("  None detected")

        print(f"\n[ Overall Compliance Risk ] "
              f"{state['compliance_rules'].get('overall_compliance_risk', '').upper()}")

    # ── architecture_agent ──────────────────────────────────────────
    print("\n=== architecture_agent ===")
    result = architecture_agent(state)
    state["architecture"] = result["architecture"]
    state["audit_log"] += result["audit_log"]

    arch = state["architecture"]

    print(f"\n[ Selected Pattern ]  {arch.get('selected_pattern').upper()}")
    print(f"  Rationale: {arch.get('pattern_rationale')}")

    print("\n[ Trade-off Matrix ]")
    for k, v in arch.get("trade_off_matrix", {}).items():
        bar = {"low": "░░░░░░", "medium": "███░░░", "high": "██████"}.get(v, "?")
        print(f"  {k:20s} {bar}  {v.upper()}")

    print("\n[ Layers ]")
    for layer in arch.get("layers", []):
        print(f"\n  {layer['name']}")
        print(f"    Responsibility : {layer['responsibility']}")
        print(f"    Tech           : {', '.join(layer.get('tech', []))}")
        print(f"    Components     : {', '.join(layer.get('components', []))}")
        if layer.get("compliance_controls"):
            print(f"    Satisfies      : {', '.join(layer.get('compliance_controls', []))}")

    print("\n[ Infrastructure ]")
    for k, v in arch.get("infrastructure", {}).items():
        print(f"  {k:15s} : {v}")

    print("\n[ Security Controls ]")
    for sc in arch.get("security_controls", []):
        print(f"  + {sc}")

    print("\n[ Compliance Gaps Addressed ]")
    for g in arch.get("gaps_addressed", []):
        print(f"  [resolved] {g}")

    if arch.get("residual_risks"):
        print("\n[ Residual Risks ]")
        for r in arch.get("residual_risks", []):
            print(f"  [!] {r}")

    # ── HITL gate 2 — loop until approved ──────────────────────────
    max_iterations = 3
    iteration      = 0
    gate2_approved = False

    while not gate2_approved and iteration < max_iterations:
        iteration += 1
        print(f"\n=== hitl_gate_2 (attempt {iteration}) ===")

        result = hitl_gate_2(state)
        state["hitl_decisions"] = result["hitl_decisions"]
        state["architecture"]   = result["architecture"]
        state["raw_input"]      = result["raw_input"]
        state["audit_log"]     += result["audit_log"]

        last = state["hitl_decisions"][-1]

        if last["choice"] == "R":
            print(f"\n>> Rejected — reason: {last['feedback']}")
            print(">> Re-running architecture_agent with feedback...")

            result = architecture_agent(state)
            state["architecture"] = result["architecture"]
            state["audit_log"]   += result["audit_log"]

            print("\n>> Updated layers:")
            for layer in state["architecture"].get("layers", []):
                print(f"   - {layer['name']} → {', '.join(layer.get('tech', []))}")

        elif last["choice"] == "M" and last.get("extra_notes"):
            print(f"\n>> Modified — constraint added: {last['extra_notes']}")
            gate2_approved = True

        else:  # A
            gate2_approved = True
            print("\n>> Approved — proceeding to codegen + explainability agents")

    if not gate2_approved:
        print(f"\n>> Max iterations ({max_iterations}) reached — escalating")
        return

    print("\n=== Full Audit Log ===")
    for entry in state["audit_log"]:
        print(f"  [{entry['timestamp']}] {entry['agent']:20s} — {entry['summary']}")


if __name__ == "__main__":
    main()

=== intent_agent ===

=== ip_guard_agent ===

=== hitl_gate_1 (attempt 1) ===

  HITL GATE 1 — INTENT + IP REVIEW

[ Intent Manifest ]
  App type : web app
  Modules  :
    - Authentication Module → FastAPI, Passlib, PyJWT
    - Dashboard Module → FastAPI, Jinja2, HTML/CSS
    - Data Persistence Layer → PostgreSQL, SQLAlchemy, Alembic
  Security constraints  : ['Password hashing with bcrypt', 'JWT token expiration', 'Secure cookie handling']
  Compliance constraints: ['None specified']
  Acceptance criteria   :
    - User can register a new account
    - User can log in and receive a session token
    - Dashboard is inaccessible to unauthenticated users
    - Data is successfully persisted in PostgreSQL

[ IP Clearance Report ]
  Overall risk : LOW
  Flagged items: none
  Recommendation: The project dependencies are all under permissive licenses; no further legal action is required for compliance.

[ Scanned Libraries ]
  [ok]     FastAPI              MIT             — Permissive licen

In [16]:
CODEGEN_SCHEMA = """
{
  "modules": [
    {
      "filename": "string — e.g. auth/router.py",
      "layer": "string — which arch layer this belongs to",
      "description": "string — what this file does",
      "rationale": "string — why it was built this way",
      "compliance_controls": ["string — which rules this file satisfies"],
      "code": "string — the actual python code"
    }
  ],
  "project_structure": ["string — full file tree e.g. app/main.py"],
  "setup_instructions": ["string — how to run the project"],
  "dependencies": ["string — pip install requirements"]
}
"""

def codegen_agent(state: DevState) -> dict:
    manifest    = state["intent_manifest"]
    arch        = state["architecture"]
    compliance  = state["compliance_rules"]
    ip          = state["ip_clearance"]

    layers              = arch.get("layers", [])
    infra               = arch.get("infrastructure", {})
    security_controls   = arch.get("security_controls", [])
    consolidated_rules  = compliance.get("consolidated_rules", [])
    human_constraints   = arch.get("human_constraints", [])

    prompt = f"""
You are a senior Python engineer generating production-grade code.

Intent Manifest:
{json.dumps(manifest, indent=2)}

Approved Architecture:
- Pattern    : {arch.get('selected_pattern')}
- Layers     : {json.dumps([l['name'] for l in layers], indent=2)}
- Infrastructure: {json.dumps(infra, indent=2)}

Security Controls to Implement:
{json.dumps(security_controls, indent=2)}

Compliance Rules to Satisfy in Code:
{json.dumps(consolidated_rules, indent=2)}

Human Constraints Added at Review:
{json.dumps(human_constraints, indent=2) if human_constraints else "None"}

IP Cleared Libraries Only:
{json.dumps([lib['name'] for lib in ip.get('scanned_libraries', [])], indent=2)}

Your tasks:
1. Generate all necessary Python files to implement the approved architecture
2. Every function must have a docstring explaining what it does and why
3. Every file must have a module-level comment mapping it to an architecture layer
4. Security controls must be implemented exactly as specified — no shortcuts
5. Compliance rules must be visible in the code as inline comments
   e.g. # [GDPR] encrypted at rest via TDE  or  # [OWASP] parameterized query
6. Use ONLY the IP-cleared libraries listed above
7. Include a main.py, requirements.txt content, and alembic setup

Respond ONLY with valid JSON matching this schema, no explanation, no markdown:
{CODEGEN_SCHEMA}
"""

    response = llm.invoke(prompt)
    code     = extract_json(response.text)

    modules        = code.get("modules", [])
    files_generated = [m["filename"] for m in modules]

    audit_entry = make_audit_entry(
        agent   = "codegen_agent",
        summary = f"Code generated — {len(modules)} files | pattern: {arch.get('selected_pattern')}",
        data    = {
            "files_generated":   files_generated,
            "dependencies":      code.get("dependencies", []),
            "pattern":           arch.get("selected_pattern"),
            "compliance_tagged": True,
        }
    )

    return {
        "generated_code": code,
        "audit_log":      [audit_entry],
    }

In [15]:
def main():
    state: DevState = {
        "raw_input":           "Build a FastAPI web app with login and dashboard using PostgreSQL",
        "intent_manifest":     None,
        "compliance_rules":    None,
        "ip_clearance":        None,
        "architecture":        None,
        "generated_code":      None,
        "explainability_docs": None,
        "security_report":     None,
        "quality_report":      None,
        "hitl_decisions":      [],
        "audit_log":           [],
        "drift_alerts":        None,
    }

    # ── agent 1 ──
    print("=== intent_agent ===")
    result = intent_agent(state)
    state["intent_manifest"] = result["intent_manifest"]
    state["audit_log"] += result["audit_log"]

    # ── agent 2 ──
    print("\n=== ip_guard_agent ===")
    result = ip_guard_agent(state)
    state["ip_clearance"] = result["ip_clearance"]
    state["audit_log"] += result["audit_log"]

    # ── HITL gate 1 ────────────────────────────────────────────────
    max_iterations = 3
    iteration      = 0
    gate1_approved = False

    while not gate1_approved and iteration < max_iterations:
        iteration += 1
        print(f"\n=== hitl_gate_1 (attempt {iteration}) ===")

        result = hitl_gate_1(state)
        state["hitl_decisions"]  = result["hitl_decisions"]
        state["intent_manifest"] = result["intent_manifest"]
        state["raw_input"]       = result["raw_input"]
        state["audit_log"]      += result["audit_log"]

        last = state["hitl_decisions"][-1]

        if last["choice"] == "R":
            print(f"\n>> Rejected — reason: {last['feedback']}")
            print(">> Re-running intent_agent with feedback...")
            result = intent_agent(state)
            state["intent_manifest"] = result["intent_manifest"]
            state["audit_log"] += result["audit_log"]

            print(">> Re-running ip_guard_agent on updated manifest...")
            result = ip_guard_agent(state)
            state["ip_clearance"] = result["ip_clearance"]
            state["audit_log"] += result["audit_log"]

            print("\n>> Updated manifest modules:")
            for m in state["intent_manifest"].get("modules", []):
                print(f"   - {m['name']} → {', '.join(m.get('tech_stack', []))}")

        elif last["choice"] == "M" and last.get("extra_notes"):
            print(f"\n>> Modified — applying: {last['extra_notes']}")
            state["raw_input"] = (
                f"{state['raw_input']}\n\n"
                f"[Human modification]: {last['extra_notes']}\n"
                f"Please update the intent manifest to reflect this change exactly."
            )
            result = intent_agent(state)
            state["intent_manifest"] = result["intent_manifest"]
            state["audit_log"] += result["audit_log"]

            result = ip_guard_agent(state)
            state["ip_clearance"] = result["ip_clearance"]
            state["audit_log"] += result["audit_log"]

            gate1_approved = True

        else:
            gate1_approved = True
            print("\n>> Approved — proceeding to compliance_agent")

    if not gate1_approved:
        print(f"\n>> Max iterations reached — escalating")
        return

    # ── compliance_agent ────────────────────────────────────────────
    print("\n=== compliance_agent ===")
    result = compliance_agent(state)
    state["compliance_rules"] = result["compliance_rules"]
    state["audit_log"] += result["audit_log"]

    print("\n[ Applicable Frameworks ]")
    for fw in state["compliance_rules"].get("applicable_frameworks", []):
        print(f"  {fw['priority'].upper():12s} {fw['name']}")
        print(f"               {fw['reason']}")

    print("\n[ Consolidated Rules ]")
    for i, r in enumerate(state["compliance_rules"].get("consolidated_rules", []), 1):
        print(f"  {i:2}. [{r['framework']}] {r['rule']}")
        print(f"      Hint: {r['implementation_hint']}")

    print("\n[ Gaps Detected ]")
    gaps = state["compliance_rules"].get("gaps", [])
    if gaps:
        for g in gaps:
            print(f"  ! {g}")
    else:
        print("  None detected")

    print(f"\n[ Overall Compliance Risk ] "
          f"{state['compliance_rules'].get('overall_compliance_risk', '').upper()}")

    # ── architecture_agent ──────────────────────────────────────────
    print("\n=== architecture_agent ===")
    result = architecture_agent(state)
    state["architecture"] = result["architecture"]
    state["audit_log"] += result["audit_log"]

    arch = state["architecture"]

    print(f"\n[ Selected Pattern ]  {arch.get('selected_pattern', '').upper()}")
    print(f"  Rationale: {arch.get('pattern_rationale')}")

    print("\n[ Trade-off Matrix ]")
    for k, v in arch.get("trade_off_matrix", {}).items():
        print(f"  {k:20s} {v.upper()}")

    print("\n[ Layers ]")
    for layer in arch.get("layers", []):
        print(f"\n  {layer['name']}")
        print(f"    Responsibility : {layer['responsibility']}")
        print(f"    Tech           : {', '.join(layer.get('tech', []))}")
        print(f"    Components     : {', '.join(layer.get('components', []))}")
        if layer.get("compliance_controls"):
            print(f"    Satisfies      : {', '.join(layer.get('compliance_controls', []))}")

    print("\n[ Infrastructure ]")
    for k, v in arch.get("infrastructure", {}).items():
        print(f"  {k:15s} : {v}")

    print("\n[ Security Controls ]")
    for sc in arch.get("security_controls", []):
        print(f"  + {sc}")

    print("\n[ Compliance Gaps Addressed ]")
    for g in arch.get("gaps_addressed", []):
        print(f"  [resolved] {g}")

    if arch.get("residual_risks"):
        print("\n[ Residual Risks ]")
        for r in arch.get("residual_risks", []):
            print(f"  [!] {r}")

    # ── HITL gate 2 ─────────────────────────────────────────────────
    max_iterations = 3
    iteration      = 0
    gate2_approved = False

    while not gate2_approved and iteration < max_iterations:
        iteration += 1
        print(f"\n=== hitl_gate_2 (attempt {iteration}) ===")

        result = hitl_gate_2(state)
        state["hitl_decisions"] = result["hitl_decisions"]
        state["architecture"]   = result["architecture"]
        state["raw_input"]      = result["raw_input"]
        state["audit_log"]     += result["audit_log"]

        last = state["hitl_decisions"][-1]

        if last["choice"] == "R":
            print(f"\n>> Rejected — reason: {last['feedback']}")
            print(">> Re-running architecture_agent with feedback...")
            result = architecture_agent(state)
            state["architecture"] = result["architecture"]
            state["audit_log"]   += result["audit_log"]

            print("\n>> Updated layers:")
            for layer in state["architecture"].get("layers", []):
                print(f"   - {layer['name']} → {', '.join(layer.get('tech', []))}")

        elif last["choice"] == "M" and last.get("extra_notes"):
            print(f"\n>> Modified — constraint added: {last['extra_notes']}")
            gate2_approved = True

        else:
            gate2_approved = True
            print("\n>> Approved — proceeding to codegen agent")

    if not gate2_approved:
        print(f"\n>> Max iterations reached — escalating")
        return

    # ── codegen_agent ───────────────────────────────────────────────
    print("\n=== codegen_agent ===")
    result = codegen_agent(state)
    state["generated_code"] = result["generated_code"]
    state["audit_log"] += result["audit_log"]

    code = state["generated_code"]

    print("\n[ Project Structure ]")
    for f in code.get("project_structure", []):
        print(f"  {f}")

    print("\n[ Generated Modules ]")
    for m in code.get("modules", []):
        print(f"\n  {m['filename']}  [{m['layer']}]")
        print(f"    {m['description']}")
        print(f"    Rationale  : {m['rationale']}")
        if m.get("compliance_controls"):
            print(f"    Satisfies  : {', '.join(m['compliance_controls'])}")

    print("\n[ Dependencies ]")
    for dep in code.get("dependencies", []):
        print(f"  {dep}")

    print("\n[ Setup Instructions ]")
    for step in code.get("setup_instructions", []):
        print(f"  → {step}")

    print("\n[ Generated Code Preview ]")
    for m in code.get("modules", []):
        print(f"\n{'='*50}")
        print(f"  FILE: {m['filename']}")
        print(f"{'='*50}")
        print(m.get("code", ""))

    # ── full audit trail ────────────────────────────────────────────
    print("\n=== Full Audit Log ===")
    for entry in state["audit_log"]:
        print(f"  [{entry['timestamp']}] {entry['agent']:20s} — {entry['summary']}")


if __name__ == "__main__":
    main()

=== intent_agent ===

=== ip_guard_agent ===

=== hitl_gate_1 (attempt 1) ===

  HITL GATE 1 — INTENT + IP REVIEW

[ Intent Manifest ]
  App type : web app
  Modules  :
    - Authentication Module → FastAPI, Passlib, PyJWT
    - Dashboard Module → FastAPI, Jinja2, HTML/CSS
    - Data Persistence Module → PostgreSQL, SQLAlchemy, Alembic
  Security constraints  : ['Password hashing with bcrypt', 'JWT token expiration', 'Secure cookie handling']
  Compliance constraints: ['None specified']
  Acceptance criteria   :
    - User can register a new account
    - User can log in and receive an authentication token
    - Dashboard is inaccessible to unauthenticated users
    - Data is successfully persisted in PostgreSQL

[ IP Clearance Report ]
  Overall risk : LOW
  Flagged items: none
  Recommendation: The current stack is composed of permissive licenses; no further legal action is required for commercial deployment.

[ Scanned Libraries ]
  [ok]     FastAPI              MIT             — Pe

In [17]:
SECURITY_RULES = {
    "hardcoded_secrets": {
        "pattern":     r'(SECRET_KEY|PASSWORD|API_KEY|TOKEN|secret|password)\s*=\s*["\'][^"\']+["\']',
        "severity":    "critical",
        "owasp":       "A02:2021 Cryptographic Failures",
        "fix":         "Move to environment variables using python-dotenv or secrets manager",
    },
    "sql_injection": {
        "pattern":     r'execute\s*\(\s*[f"\'].*\{',
        "severity":    "critical",
        "owasp":       "A03:2021 Injection",
        "fix":         "Use SQLAlchemy ORM or parameterized queries only",
    },
    "debug_mode": {
        "pattern":     r'DEBUG\s*=\s*True|reload\s*=\s*True',
        "severity":    "high",
        "owasp":       "A05:2021 Security Misconfiguration",
        "fix":         "Disable debug/reload in production — use environment flag",
    },
    "http_not_https": {
        "pattern":     r'http://(?!localhost|127\.0\.0\.1)',
        "severity":    "high",
        "owasp":       "A02:2021 Cryptographic Failures",
        "fix":         "Use HTTPS for all external URLs",
    },
    "bare_except": {
        "pattern":     r'except\s*:',
        "severity":    "medium",
        "owasp":       "A09:2021 Security Logging and Monitoring Failures",
        "fix":         "Catch specific exceptions — bare except hides security errors",
    },
    "missing_auth_dependency": {
        "pattern":     r'@router\.(get|post|put|delete)\s*\([^)]*\)\s*\nasync def(?!.*Depends)',
        "severity":    "high",
        "owasp":       "A01:2021 Broken Access Control",
        "fix":         "Add FastAPI Depends(get_current_user) to protected routes",
    },
    "unlicensed_import": {
        "pattern":     None,   # handled separately via ip_clearance check
        "severity":    "high",
        "owasp":       "IP Compliance",
        "fix":         "Use only IP-cleared libraries",
    },
}

SECURITY_SCHEMA = """
{
  "findings": [
    {
      "filename":    "string",
      "rule":        "string — which security rule was violated",
      "severity":    "critical | high | medium | low",
      "owasp_ref":   "string — OWASP category",
      "line_hint":   "string — the problematic code snippet",
      "fix":         "string — how to fix it"
    }
  ],
  "unlicensed_imports": ["string — any imports not in ip_clearance"],
  "compliance_tag_coverage": {
    "files_with_tags":    "number — files containing compliance comments",
    "files_without_tags": ["string — filenames missing compliance comments"],
    "coverage_percent":   "number"
  },
  "overall_security_risk": "critical | high | medium | low",
  "passed":                "boolean — true only if no critical findings",
  "summary":               "string — one paragraph summary"
}
"""

def run_local_security_scan(modules: list, ip_clearance: dict) -> dict:

    local_findings = []
    cleared_libs   = {
        lib["name"].lower()
        for lib in ip_clearance.get("scanned_libraries", [])
    }

    for module in modules:
        filename = module.get("filename", "unknown")
        code     = module.get("code", "")

        for rule_name, rule in SECURITY_RULES.items():
            if rule["pattern"] is None:
                continue
            matches = re.findall(rule["pattern"], code, re.MULTILINE)
            if matches:
                local_findings.append({
                    "filename":  filename,
                    "rule":      rule_name,
                    "severity":  rule["severity"],
                    "owasp_ref": rule["owasp"],
                    "line_hint": str(matches[0])[:120],
                    "fix":       rule["fix"],
                    "source":    "local_scan",
                })

        imports = re.findall(r'^(?:import|from)\s+(\w+)', code, re.MULTILINE)
        for imp in imports:
            imp_lower = imp.lower()
            stdlib = {
                "os", "sys", "re", "json", "datetime", "typing",
                "pathlib", "hashlib", "hmac", "secrets", "logging",
                "functools", "itertools", "collections", "abc", "enum","pydantic", "starlette", "asyncpg", "sqlalchemy", "alembic", "pytest", "httpx", "uvicorn", "fastapi"
            }
            if imp_lower not in stdlib and imp_lower not in cleared_libs:
                local_findings.append({
                    "filename":  filename,
                    "rule":      "unlicensed_import",
                    "severity":  "high",
                    "owasp_ref": "IP Compliance",
                    "line_hint": f"import {imp}",
                    "fix":       f"'{imp}' was not IP-cleared — verify license before use",
                    "source":    "local_scan",
                })

    return local_findings


def check_compliance_tag_coverage(modules: list) -> dict:
    """Check every file has compliance inline comments."""
    files_with    = []
    files_without = []

    for module in modules:
        code     = module.get("code", "")
        filename = module.get("filename", "unknown")
        has_tags = bool(re.search(r'#\s*\[(GDPR|OWASP|HIPAA|PCI)', code))
        if has_tags:
            files_with.append(filename)
        else:
            files_without.append(filename)

    total    = len(modules)
    coverage = round((len(files_with) / total * 100) if total > 0 else 0, 1)

    return {
        "files_with_tags":    len(files_with),
        "files_without_tags": files_without,
        "coverage_percent":   coverage,
    }


def security_agent(state: DevState) -> dict:
    code_state   = state["generated_code"]
    ip_clearance = state["ip_clearance"]
    modules      = code_state.get("modules", [])

    local_findings       = run_local_security_scan(modules, ip_clearance)
    tag_coverage         = check_compliance_tag_coverage(modules)

    code_dump = "\n\n".join([
        f"### {m['filename']}\n{m['code']}"
        for m in modules
    ])

    prompt = f"""
You are a senior application security engineer performing a code review.

Review the following generated Python code for security vulnerabilities.

Code:
{code_dump}

Local scan already detected these issues (include them in your findings):
{json.dumps(local_findings, indent=2)}

IP Cleared Libraries:
{json.dumps([lib['name'] for lib in ip_clearance.get('scanned_libraries', [])], indent=2)}

Compliance Tag Coverage:
{json.dumps(tag_coverage, indent=2)}

Check specifically for:
1. Hardcoded secrets, keys, or passwords
2. SQL injection vulnerabilities
3. Missing authentication on protected routes
4. Insecure JWT configuration (weak algo, no expiry)
5. Missing input validation
6. Debug mode enabled in production
7. Any imports not in the IP-cleared list
8. Missing or incorrect compliance inline comments
9. Insecure cookie configuration
10. Any other OWASP Top 10 violations

Severity levels:
- critical : exploitable immediately, must fix before deploy
- high     : significant risk, should fix before deploy
- medium   : moderate risk, fix soon
- low      : minor issue, fix when possible

Set "passed" to true ONLY if there are zero critical findings.

Respond ONLY with valid JSON matching this schema, no explanation, no markdown:
{SECURITY_SCHEMA}
"""

    response = llm.invoke(prompt)
    report   = extract_json(response.text)

    existing_keys = {
        (f["filename"], f["rule"])
        for f in report.get("findings", [])
    }
    for lf in local_findings:
        key = (lf["filename"], lf["rule"])
        if key not in existing_keys:
            report.setdefault("findings", []).append(lf)

    report["compliance_tag_coverage"] = tag_coverage

    findings        = report.get("findings", [])
    critical_count  = sum(1 for f in findings if f["severity"] == "critical")
    high_count      = sum(1 for f in findings if f["severity"] == "high")
    passed          = report.get("passed", False)
    overall_risk    = report.get("overall_security_risk", "unknown")

    audit_entry = make_audit_entry(
        agent   = "security_agent",
        summary = (
            f"Security scan complete — risk: {overall_risk} | "
            f"findings: {len(findings)} "
            f"(critical: {critical_count}, high: {high_count}) | "
            f"passed: {passed}"
        ),
        data    = {
            "total_findings":  len(findings),
            "critical":        critical_count,
            "high":            high_count,
            "passed":          passed,
            "overall_risk":    overall_risk,
            "tag_coverage":    tag_coverage["coverage_percent"],
        }
    )

    return {
        "security_report": report,
        "audit_log":       [audit_entry],
    }

In [17]:
def main():
    state: DevState = {
        "raw_input":           "Build a FastAPI web app with login and dashboard using PostgreSQL",
        "intent_manifest":     None,
        "compliance_rules":    None,
        "ip_clearance":        None,
        "architecture":        None,
        "generated_code":      None,
        "explainability_docs": None,
        "security_report":     None,
        "quality_report":      None,
        "hitl_decisions":      [],
        "audit_log":           [],
        "drift_alerts":        None,
    }

    # ── agent 1 ──
    print("=== intent_agent ===")
    result = intent_agent(state)
    state["intent_manifest"] = result["intent_manifest"]
    state["audit_log"] += result["audit_log"]

    # ── agent 2 ──
    print("\n=== ip_guard_agent ===")
    result = ip_guard_agent(state)
    state["ip_clearance"] = result["ip_clearance"]
    state["audit_log"] += result["audit_log"]

    # ── HITL gate 1 ────────────────────────────────────────────────
    max_iterations = 3
    iteration      = 0
    gate1_approved = False

    while not gate1_approved and iteration < max_iterations:
        iteration += 1
        print(f"\n=== hitl_gate_1 (attempt {iteration}) ===")

        result = hitl_gate_1(state)
        state["hitl_decisions"]  = result["hitl_decisions"]
        state["intent_manifest"] = result["intent_manifest"]
        state["raw_input"]       = result["raw_input"]
        state["audit_log"]      += result["audit_log"]

        last = state["hitl_decisions"][-1]

        if last["choice"] == "R":
            print(f"\n>> Rejected — reason: {last['feedback']}")
            print(">> Re-running intent_agent with feedback...")
            result = intent_agent(state)
            state["intent_manifest"] = result["intent_manifest"]
            state["audit_log"] += result["audit_log"]

            print(">> Re-running ip_guard_agent on updated manifest...")
            result = ip_guard_agent(state)
            state["ip_clearance"] = result["ip_clearance"]
            state["audit_log"] += result["audit_log"]

            print("\n>> Updated manifest modules:")
            for m in state["intent_manifest"].get("modules", []):
                print(f"   - {m['name']} → {', '.join(m.get('tech_stack', []))}")

        elif last["choice"] == "M" and last.get("extra_notes"):
            print(f"\n>> Modified — applying: {last['extra_notes']}")
            state["raw_input"] = (
                f"{state['raw_input']}\n\n"
                f"[Human modification]: {last['extra_notes']}\n"
                f"Please update the intent manifest to reflect this change exactly."
            )
            result = intent_agent(state)
            state["intent_manifest"] = result["intent_manifest"]
            state["audit_log"] += result["audit_log"]

            result = ip_guard_agent(state)
            state["ip_clearance"] = result["ip_clearance"]
            state["audit_log"] += result["audit_log"]

            gate1_approved = True

        else:
            gate1_approved = True
            print("\n>> Approved — proceeding to compliance_agent")

    if not gate1_approved:
        print(f"\n>> Max iterations reached — escalating")
        return

    # ── compliance_agent ────────────────────────────────────────────
    print("\n=== compliance_agent ===")
    result = compliance_agent(state)
    state["compliance_rules"] = result["compliance_rules"]
    state["audit_log"] += result["audit_log"]

    print("\n[ Applicable Frameworks ]")
    for fw in state["compliance_rules"].get("applicable_frameworks", []):
        print(f"  {fw['priority'].upper():12s} {fw['name']}")
        print(f"               {fw['reason']}")

    print("\n[ Consolidated Rules ]")
    for i, r in enumerate(state["compliance_rules"].get("consolidated_rules", []), 1):
        print(f"  {i:2}. [{r['framework']}] {r['rule']}")
        print(f"      Hint: {r['implementation_hint']}")

    print("\n[ Gaps Detected ]")
    gaps = state["compliance_rules"].get("gaps", [])
    if gaps:
        for g in gaps:
            print(f"  ! {g}")
    else:
        print("  None detected")

    print(f"\n[ Overall Compliance Risk ] "
          f"{state['compliance_rules'].get('overall_compliance_risk', '').upper()}")

    # ── architecture_agent ──────────────────────────────────────────
    print("\n=== architecture_agent ===")
    result = architecture_agent(state)
    state["architecture"] = result["architecture"]
    state["audit_log"] += result["audit_log"]

    arch = state["architecture"]

    print(f"\n[ Selected Pattern ]  {arch.get('selected_pattern', '').upper()}")
    print(f"  Rationale: {arch.get('pattern_rationale')}")

    print("\n[ Trade-off Matrix ]")
    for k, v in arch.get("trade_off_matrix", {}).items():
        print(f"  {k:20s} {v.upper()}")

    print("\n[ Layers ]")
    for layer in arch.get("layers", []):
        print(f"\n  {layer['name']}")
        print(f"    Responsibility : {layer['responsibility']}")
        print(f"    Tech           : {', '.join(layer.get('tech', []))}")
        print(f"    Components     : {', '.join(layer.get('components', []))}")
        if layer.get("compliance_controls"):
            print(f"    Satisfies      : {', '.join(layer.get('compliance_controls', []))}")

    print("\n[ Infrastructure ]")
    for k, v in arch.get("infrastructure", {}).items():
        print(f"  {k:15s} : {v}")

    print("\n[ Security Controls ]")
    for sc in arch.get("security_controls", []):
        print(f"  + {sc}")

    print("\n[ Compliance Gaps Addressed ]")
    for g in arch.get("gaps_addressed", []):
        print(f"  [resolved] {g}")

    if arch.get("residual_risks"):
        print("\n[ Residual Risks ]")
        for r in arch.get("residual_risks", []):
            print(f"  [!] {r}")

    # ── HITL gate 2 ─────────────────────────────────────────────────
    max_iterations = 3
    iteration      = 0
    gate2_approved = False

    while not gate2_approved and iteration < max_iterations:
        iteration += 1
        print(f"\n=== hitl_gate_2 (attempt {iteration}) ===")

        result = hitl_gate_2(state)
        state["hitl_decisions"] = result["hitl_decisions"]
        state["architecture"]   = result["architecture"]
        state["raw_input"]      = result["raw_input"]
        state["audit_log"]     += result["audit_log"]

        last = state["hitl_decisions"][-1]

        if last["choice"] == "R":
            print(f"\n>> Rejected — reason: {last['feedback']}")
            print(">> Re-running architecture_agent with feedback...")
            result = architecture_agent(state)
            state["architecture"] = result["architecture"]
            state["audit_log"]   += result["audit_log"]

            print("\n>> Updated layers:")
            for layer in state["architecture"].get("layers", []):
                print(f"   - {layer['name']} → {', '.join(layer.get('tech', []))}")

        elif last["choice"] == "M" and last.get("extra_notes"):
            print(f"\n>> Modified — constraint added: {last['extra_notes']}")
            gate2_approved = True

        else:
            gate2_approved = True
            print("\n>> Approved — proceeding to codegen agent")

    if not gate2_approved:
        print(f"\n>> Max iterations reached — escalating")
        return

    # ── codegen_agent ───────────────────────────────────────────────
    print("\n=== codegen_agent ===")
    result = codegen_agent(state)
    state["generated_code"] = result["generated_code"]
    state["audit_log"] += result["audit_log"]

    code = state["generated_code"]

    print("\n[ Project Structure ]")
    for f in code.get("project_structure", []):
        print(f"  {f}")

    print("\n[ Generated Modules ]")
    for m in code.get("modules", []):
        print(f"\n  {m['filename']}  [{m['layer']}]")
        print(f"    {m['description']}")
        print(f"    Rationale  : {m['rationale']}")
        if m.get("compliance_controls"):
            print(f"    Satisfies  : {', '.join(m['compliance_controls'])}")

    print("\n[ Dependencies ]")
    for dep in code.get("dependencies", []):
        print(f"  {dep}")

    print("\n[ Setup Instructions ]")
    for step in code.get("setup_instructions", []):
        print(f"  → {step}")

    print("\n[ Generated Code Preview ]")
    for m in code.get("modules", []):
        print(f"\n{'='*50}")
        print(f"  FILE: {m['filename']}")
        print(f"{'='*50}")
        print(m.get("code", ""))
    
    # ── security_agent ──────────────────────────────────────────────
    print("\n=== security_agent ===")
    result = security_agent(state)
    state["security_report"] = result["security_report"]
    state["audit_log"] += result["audit_log"]

    sec = state["security_report"]

    print(f"\n[ Overall Security Risk ]  {sec.get('overall_security_risk', '').upper()}")
    print(f"[ Passed ]  {sec.get('passed')}")

    print("\n[ Findings ]")
    findings = sec.get("findings", [])
    if findings:
        for f in findings:
            sev = f.get("severity", "").upper()
            print(f"\n  [{sev}] {f['filename']}")
            print(f"    Rule     : {f['rule']}")
            print(f"    OWASP    : {f['owasp_ref']}")
            print(f"    Code     : {f['line_hint']}")
            print(f"    Fix      : {f['fix']}")
    else:
        print("  No findings")

    print("\n[ Compliance Tag Coverage ]")
    cov = sec.get("compliance_tag_coverage", {})
    print(f"  Coverage : {cov.get('coverage_percent')}%")
    if cov.get("files_without_tags"):
        print(f"  Missing tags in: {', '.join(cov['files_without_tags'])}")

    if sec.get("unlicensed_imports"):
        print("\n[ Unlicensed Imports ]")
        for u in sec.get("unlicensed_imports", []):
            print(f"  [!] {u}")

    print(f"\n[ Summary ]\n  {sec.get('summary')}")

    # route based on security outcome
    if not sec.get("passed"):
        print("\n>> Security scan FAILED — critical findings must be fixed")
        print(">> In LangGraph this routes back to codegen_agent")
        print(">> For now printing what would be fixed...")
    else:
        print("\n>> Security scan PASSED — proceeding to quality_agent")

    print("\n=== Full Audit Log ===")
    for entry in state["audit_log"]:
        print(f"  [{entry['timestamp']}] {entry['agent']:20s} — {entry['summary']}")


if __name__ == "__main__":
    main()

=== intent_agent ===

=== ip_guard_agent ===

=== hitl_gate_1 (attempt 1) ===

  HITL GATE 1 — INTENT + IP REVIEW

[ Intent Manifest ]
  App type : web app
  Modules  :
    - Authentication Module → FastAPI, Passlib, PyJWT
    - Dashboard Module → FastAPI, Jinja2, HTML/CSS
    - Data Persistence Module → PostgreSQL, SQLAlchemy, Alembic
  Security constraints  : ['Password hashing with Bcrypt', 'JWT token expiration', 'CORS policy configuration']
  Compliance constraints: ['None specified']
  Acceptance criteria   :
    - User can register a new account
    - User can log in and receive an authentication token
    - Dashboard is inaccessible to unauthenticated users
    - Data is successfully persisted in PostgreSQL

[ IP Clearance Report ]
  Overall risk : LOW
  Flagged items: none
  Recommendation: The current stack consists entirely of permissive licenses; no further legal action is required for commercial deployment.

[ Scanned Libraries ]
  [ok]     FastAPI              MIT        

In [18]:
EXPLAINABILITY_SCHEMA = """
{
  "decision_log": [
    {
      "decision_point": "string — e.g. Architecture Pattern Selection",
      "what_was_decided": "string — the actual decision made",
      "why": "string — plain English reasoning",
      "alternatives_considered": ["string — other options that were evaluated"],
      "trade_offs_accepted": ["string — what was given up"],
      "constraint_satisfied": ["string — which intent/compliance constraint this serves"]
    }
  ],
  "module_explanations": [
    {
      "filename": "string",
      "purpose": "string — what this file does in plain English",
      "key_decisions": ["string — notable implementation choices"],
      "compliance_mapping": ["string — which rules this file satisfies and how"]
    }
  ],
  "glossary": [
    {
      "term": "string — technical term used in the project",
      "plain_english": "string — what it means for a non-technical stakeholder"
    }
  ],
  "audit_narrative": "string — a single paragraph telling the full story of how this system was built, what decisions were made, and why — written for a regulator or auditor"
}
"""

def explainability_agent(state: DevState) -> dict:
    manifest    = state["intent_manifest"]
    arch        = state["architecture"]
    compliance  = state["compliance_rules"]
    code        = state["generated_code"]
    security    = state["security_report"]
    decisions   = state["hitl_decisions"]

    hitl_trail = []
    for d in decisions:
        hitl_trail.append({
            "gate":     d.get("gate"),
            "choice":   d.get("choice"),
            "approver": d.get("approver"),
            "notes":    d.get("extra_notes") or d.get("feedback") or "none",
            "risks_acknowledged": d.get("risk_acknowledged", False),
        })

    prompt = f"""
You are a technical documentation specialist and compliance explainer.

Your job is to produce a complete explainability report for a software system
that was designed and built by an AI pipeline. This report must be readable by:
- Non-technical business stakeholders
- Regulators and auditors
- Future developers joining the project

Here is everything the pipeline produced:

Original User Intent:
{state['raw_input']}

Intent Manifest:
{json.dumps(manifest, indent=2)}

Architecture Decisions:
- Pattern   : {arch.get('selected_pattern')}
- Rationale : {arch.get('pattern_rationale')}
- Layers    : {json.dumps([l['name'] for l in arch.get('layers', [])], indent=2)}
- Residual Risks: {json.dumps(arch.get('residual_risks', []), indent=2)}

Compliance Frameworks Applied:
{json.dumps([f['name'] for f in compliance.get('applicable_frameworks', [])], indent=2)}

Compliance Rules Enforced:
{json.dumps(compliance.get('consolidated_rules', []), indent=2)}

Generated Files:
{json.dumps([{
    'filename': m['filename'],
    'layer': m['layer'],
    'description': m['description'],
    'compliance_controls': m.get('compliance_controls', [])
} for m in code.get('modules', [])], indent=2)}

Security Findings:
{json.dumps(security.get('findings', []), indent=2)}

Human Review Trail:
{json.dumps(hitl_trail, indent=2)}

Your tasks:
1. Document every major decision point with plain English reasoning
2. Explain each generated file to a non-technical audience
3. Build a glossary of technical terms used
4. Write an audit narrative — one paragraph that tells the full story
   of how this system was designed, reviewed, and built

Respond ONLY with valid JSON matching this schema, no explanation, no markdown:
{EXPLAINABILITY_SCHEMA}
"""

    response = llm.invoke(prompt)
    docs     = extract_json(response.text)

    decisions_documented = len(docs.get("decision_log", []))
    modules_explained    = len(docs.get("module_explanations", []))
    glossary_terms       = len(docs.get("glossary", []))

    audit_entry = make_audit_entry(
        agent   = "explainability_agent",
        summary = (
            f"Explainability docs generated — "
            f"{decisions_documented} decisions | "
            f"{modules_explained} modules | "
            f"{glossary_terms} glossary terms"
        ),
        data    = {
            "decisions_documented": decisions_documented,
            "modules_explained":    modules_explained,
            "glossary_terms":       glossary_terms,
            "audit_narrative_len":  len(docs.get("audit_narrative", "")),
        }
    )

    return {
        "explainability_docs": docs,
        "audit_log":           [audit_entry],
    }

In [19]:
def main():
    state: DevState = {
        "raw_input":           "Build a FastAPI web app with login and dashboard using PostgreSQL",
        "intent_manifest":     None,
        "compliance_rules":    None,
        "ip_clearance":        None,
        "architecture":        None,
        "generated_code":      None,
        "explainability_docs": None,
        "security_report":     None,
        "quality_report":      None,
        "hitl_decisions":      [],
        "audit_log":           [],
        "drift_alerts":        None,
    }

    # ── agent 1 ──
    print("=== intent_agent ===")
    result = intent_agent(state)
    state["intent_manifest"] = result["intent_manifest"]
    state["audit_log"] += result["audit_log"]

    # ── agent 2 ──
    print("\n=== ip_guard_agent ===")
    result = ip_guard_agent(state)
    state["ip_clearance"] = result["ip_clearance"]
    state["audit_log"] += result["audit_log"]

    # ── HITL gate 1 ────────────────────────────────────────────────
    max_iterations = 3
    iteration      = 0
    gate1_approved = False

    while not gate1_approved and iteration < max_iterations:
        iteration += 1
        print(f"\n=== hitl_gate_1 (attempt {iteration}) ===")

        result = hitl_gate_1(state)
        state["hitl_decisions"]  = result["hitl_decisions"]
        state["intent_manifest"] = result["intent_manifest"]
        state["raw_input"]       = result["raw_input"]
        state["audit_log"]      += result["audit_log"]

        last = state["hitl_decisions"][-1]

        if last["choice"] == "R":
            print(f"\n>> Rejected — reason: {last['feedback']}")
            print(">> Re-running intent_agent with feedback...")
            result = intent_agent(state)
            state["intent_manifest"] = result["intent_manifest"]
            state["audit_log"] += result["audit_log"]

            print(">> Re-running ip_guard_agent on updated manifest...")
            result = ip_guard_agent(state)
            state["ip_clearance"] = result["ip_clearance"]
            state["audit_log"] += result["audit_log"]

            print("\n>> Updated manifest modules:")
            for m in state["intent_manifest"].get("modules", []):
                print(f"   - {m['name']} → {', '.join(m.get('tech_stack', []))}")

        elif last["choice"] == "M" and last.get("extra_notes"):
            print(f"\n>> Modified — applying: {last['extra_notes']}")
            state["raw_input"] = (
                f"{state['raw_input']}\n\n"
                f"[Human modification]: {last['extra_notes']}\n"
                f"Please update the intent manifest to reflect this change exactly."
            )
            result = intent_agent(state)
            state["intent_manifest"] = result["intent_manifest"]
            state["audit_log"] += result["audit_log"]

            result = ip_guard_agent(state)
            state["ip_clearance"] = result["ip_clearance"]
            state["audit_log"] += result["audit_log"]

            gate1_approved = True

        else:
            gate1_approved = True
            print("\n>> Approved — proceeding to compliance_agent")

    if not gate1_approved:
        print(f"\n>> Max iterations reached — escalating")
        return

    # ── compliance_agent ────────────────────────────────────────────
    print("\n=== compliance_agent ===")
    result = compliance_agent(state)
    state["compliance_rules"] = result["compliance_rules"]
    state["audit_log"] += result["audit_log"]

    print("\n[ Applicable Frameworks ]")
    for fw in state["compliance_rules"].get("applicable_frameworks", []):
        print(f"  {fw['priority'].upper():12s} {fw['name']}")
        print(f"               {fw['reason']}")

    print("\n[ Consolidated Rules ]")
    for i, r in enumerate(state["compliance_rules"].get("consolidated_rules", []), 1):
        print(f"  {i:2}. [{r['framework']}] {r['rule']}")
        print(f"      Hint: {r['implementation_hint']}")

    print("\n[ Gaps Detected ]")
    gaps = state["compliance_rules"].get("gaps", [])
    if gaps:
        for g in gaps:
            print(f"  ! {g}")
    else:
        print("  None detected")

    print(f"\n[ Overall Compliance Risk ] "
          f"{state['compliance_rules'].get('overall_compliance_risk', '').upper()}")

    # ── architecture_agent ──────────────────────────────────────────
    print("\n=== architecture_agent ===")
    result = architecture_agent(state)
    state["architecture"] = result["architecture"]
    state["audit_log"] += result["audit_log"]

    arch = state["architecture"]

    print(f"\n[ Selected Pattern ]  {arch.get('selected_pattern', '').upper()}")
    print(f"  Rationale: {arch.get('pattern_rationale')}")

    print("\n[ Trade-off Matrix ]")
    for k, v in arch.get("trade_off_matrix", {}).items():
        print(f"  {k:20s} {v.upper()}")

    print("\n[ Layers ]")
    for layer in arch.get("layers", []):
        print(f"\n  {layer['name']}")
        print(f"    Responsibility : {layer['responsibility']}")
        print(f"    Tech           : {', '.join(layer.get('tech', []))}")
        print(f"    Components     : {', '.join(layer.get('components', []))}")
        if layer.get("compliance_controls"):
            print(f"    Satisfies      : {', '.join(layer.get('compliance_controls', []))}")

    print("\n[ Infrastructure ]")
    for k, v in arch.get("infrastructure", {}).items():
        print(f"  {k:15s} : {v}")

    print("\n[ Security Controls ]")
    for sc in arch.get("security_controls", []):
        print(f"  + {sc}")

    print("\n[ Compliance Gaps Addressed ]")
    for g in arch.get("gaps_addressed", []):
        print(f"  [resolved] {g}")

    if arch.get("residual_risks"):
        print("\n[ Residual Risks ]")
        for r in arch.get("residual_risks", []):
            print(f"  [!] {r}")

    # ── HITL gate 2 ─────────────────────────────────────────────────
    max_iterations = 3
    iteration      = 0
    gate2_approved = False

    while not gate2_approved and iteration < max_iterations:
        iteration += 1
        print(f"\n=== hitl_gate_2 (attempt {iteration}) ===")

        result = hitl_gate_2(state)
        state["hitl_decisions"] = result["hitl_decisions"]
        state["architecture"]   = result["architecture"]
        state["raw_input"]      = result["raw_input"]
        state["audit_log"]     += result["audit_log"]

        last = state["hitl_decisions"][-1]

        if last["choice"] == "R":
            print(f"\n>> Rejected — reason: {last['feedback']}")
            print(">> Re-running architecture_agent with feedback...")
            result = architecture_agent(state)
            state["architecture"] = result["architecture"]
            state["audit_log"]   += result["audit_log"]

            print("\n>> Updated layers:")
            for layer in state["architecture"].get("layers", []):
                print(f"   - {layer['name']} → {', '.join(layer.get('tech', []))}")

        elif last["choice"] == "M" and last.get("extra_notes"):
            print(f"\n>> Modified — constraint added: {last['extra_notes']}")
            gate2_approved = True

        else:
            gate2_approved = True
            print("\n>> Approved — proceeding to codegen agent")

    if not gate2_approved:
        print(f"\n>> Max iterations reached — escalating")
        return

    # ── codegen_agent ───────────────────────────────────────────────
    print("\n=== codegen_agent ===")
    result = codegen_agent(state)
    state["generated_code"] = result["generated_code"]
    state["audit_log"] += result["audit_log"]

    code = state["generated_code"]

    print("\n[ Project Structure ]")
    for f in code.get("project_structure", []):
        print(f"  {f}")

    print("\n[ Generated Modules ]")
    for m in code.get("modules", []):
        print(f"\n  {m['filename']}  [{m['layer']}]")
        print(f"    {m['description']}")
        print(f"    Rationale  : {m['rationale']}")
        if m.get("compliance_controls"):
            print(f"    Satisfies  : {', '.join(m['compliance_controls'])}")

    print("\n[ Dependencies ]")
    for dep in code.get("dependencies", []):
        print(f"  {dep}")

    print("\n[ Setup Instructions ]")
    for step in code.get("setup_instructions", []):
        print(f"  → {step}")

    print("\n[ Generated Code Preview ]")
    for m in code.get("modules", []):
        print(f"\n{'='*50}")
        print(f"  FILE: {m['filename']}")
        print(f"{'='*50}")
        print(m.get("code", ""))
    
    # ── security_agent ──────────────────────────────────────────────
    print("\n=== security_agent ===")
    result = security_agent(state)
    state["security_report"] = result["security_report"]
    state["audit_log"] += result["audit_log"]

    sec = state["security_report"]

    print(f"\n[ Overall Security Risk ]  {sec.get('overall_security_risk', '').upper()}")
    print(f"[ Passed ]  {sec.get('passed')}")

    print("\n[ Findings ]")
    findings = sec.get("findings", [])
    if findings:
        for f in findings:
            sev = f.get("severity", "").upper()
            print(f"\n  [{sev}] {f['filename']}")
            print(f"    Rule     : {f['rule']}")
            print(f"    OWASP    : {f['owasp_ref']}")
            print(f"    Code     : {f['line_hint']}")
            print(f"    Fix      : {f['fix']}")
    else:
        print("  No findings")

    print("\n[ Compliance Tag Coverage ]")
    cov = sec.get("compliance_tag_coverage", {})
    print(f"  Coverage : {cov.get('coverage_percent')}%")
    if cov.get("files_without_tags"):
        print(f"  Missing tags in: {', '.join(cov['files_without_tags'])}")

    if sec.get("unlicensed_imports"):
        print("\n[ Unlicensed Imports ]")
        for u in sec.get("unlicensed_imports", []):
            print(f"  [!] {u}")

    print(f"\n[ Summary ]\n  {sec.get('summary')}")

    # route based on security outcome
    if not sec.get("passed"):
        print("\n>> Security scan FAILED — critical findings must be fixed")
        print(">> In LangGraph this routes back to codegen_agent")
        print(">> For now printing what would be fixed...")
    else:
        print("\n>> Security scan PASSED — proceeding to quality_agent")

    # ── explainability_agent ────────────────────────────────────────
    print("\n=== explainability_agent ===")
    result = explainability_agent(state)
    state["explainability_docs"] = result["explainability_docs"]
    state["audit_log"] += result["audit_log"]

    docs = state["explainability_docs"]

    print("\n[ Decision Log ]")
    for d in docs.get("decision_log", []):
        print(f"\n  Decision : {d['decision_point']}")
        print(f"  What     : {d['what_was_decided']}")
        print(f"  Why      : {d['why']}")
        if d.get("alternatives_considered"):
            print(f"  Alternatives : {', '.join(d['alternatives_considered'])}")
        if d.get("trade_offs_accepted"):
            print(f"  Trade-offs   : {', '.join(d['trade_offs_accepted'])}")
        if d.get("constraint_satisfied"):
            print(f"  Satisfies    : {', '.join(d['constraint_satisfied'])}")

    print("\n[ Module Explanations ]")
    for m in docs.get("module_explanations", []):
        print(f"\n  {m['filename']}")
        print(f"  Purpose  : {m['purpose']}")
        if m.get("key_decisions"):
            print(f"  Key decisions:")
            for kd in m["key_decisions"]:
                print(f"    - {kd}")
        if m.get("compliance_mapping"):
            print(f"  Compliance:")
            for cm in m["compliance_mapping"]:
                print(f"    - {cm}")

    print("\n[ Glossary ]")
    for g in docs.get("glossary", []):
        print(f"  {g['term']:25s} — {g['plain_english']}")

    print("\n[ Audit Narrative ]")
    print(f"  {docs.get('audit_narrative', '')}")

    print("\n=== Full Audit Log ===")
    for entry in state["audit_log"]:
        print(f"  [{entry['timestamp']}] {entry['agent']:20s} — {entry['summary']}")


if __name__ == "__main__":
    main()

=== intent_agent ===

=== ip_guard_agent ===

=== hitl_gate_1 (attempt 1) ===

  HITL GATE 1 — INTENT + IP REVIEW

[ Intent Manifest ]
  App type : web app
  Modules  :
    - Authentication Module → FastAPI, Passlib, PyJWT
    - Dashboard Module → FastAPI, Jinja2, HTML/CSS
    - Data Persistence Module → PostgreSQL, SQLAlchemy, Alembic
  Security constraints  : ['Password hashing with bcrypt', 'JWT token expiration', 'Secure cookie handling']
  Compliance constraints: ['None specified']
  Acceptance criteria   :
    - User can register and log in successfully
    - Dashboard is inaccessible without valid authentication
    - Data is persisted in a PostgreSQL database
    - API endpoints are documented via OpenAPI/Swagger

[ IP Clearance Report ]
  Overall risk : LOW
  Flagged items: none
  Recommendation: All identified libraries are under permissive licenses; no further action is required for compliance.

[ Scanned Libraries ]
  [ok]     FastAPI              MIT             — Permissi

In [19]:
QUALITY_SCHEMA = """
{
  "test_results": [
    {
      "filename":    "string — file being tested",
      "test_name":   "string — what was tested",
      "status":      "pass | fail | warning",
      "detail":      "string — what was found"
    }
  ],
  "acceptance_criteria_check": [
    {
      "criterion":   "string — from intent manifest",
      "status":      "met | not_met | partial",
      "evidence":    "string — which file/function satisfies this"
    }
  ],
  "code_quality": {
    "has_docstrings":         "boolean",
    "has_type_hints":         "boolean",
    "has_error_handling":     "boolean",
    "has_async_support":      "boolean",
    "missing_docstrings_in":  ["string — filenames"],
    "missing_error_handling_in": ["string — filenames"]
  },
  "security_integration": {
    "security_findings_addressed": "boolean",
    "critical_blockers":           ["string — unresolved critical findings"],
    "ready_for_deploy":            "boolean"
  },
  "overall_quality_score":  "number — 0 to 100",
  "passed":                 "boolean — true only if score >= 70 and no critical blockers",
  "recommendations":        ["string — specific improvements to make"],
  "summary":                "string — one paragraph quality assessment"
}
"""

def run_local_quality_checks(state: DevState) -> dict:
    """
    Fast local checks before hitting the LLM.
    Checks docstrings, type hints, error handling, async support.
    """
    modules        = state["generated_code"].get("modules", [])
    manifest       = state["intent_manifest"]
    security       = state["security_report"]

    local_results  = []
    quality_issues = {
        "missing_docstrings":      [],
        "missing_type_hints":      [],
        "missing_error_handling":  [],
        "missing_async":           [],
    }

    for module in modules:
        filename = module.get("filename", "unknown")
        code     = module.get("code", "")

        # check docstrings
        has_docstring = '"""' in code or "'''" in code
        if not has_docstring:
            quality_issues["missing_docstrings"].append(filename)
            local_results.append({
                "filename":  filename,
                "test_name": "docstring_check",
                "status":    "warning",
                "detail":    "No docstrings found — functions lack documentation",
            })
        else:
            local_results.append({
                "filename":  filename,
                "test_name": "docstring_check",
                "status":    "pass",
                "detail":    "Docstrings present",
            })

        # check type hints
        has_type_hints = "->" in code or ": str" in code or ": dict" in code
        if not has_type_hints:
            quality_issues["missing_type_hints"].append(filename)
            local_results.append({
                "filename":  filename,
                "test_name": "type_hint_check",
                "status":    "warning",
                "detail":    "No type hints detected",
            })
        else:
            local_results.append({
                "filename":  filename,
                "test_name": "type_hint_check",
                "status":    "pass",
                "detail":    "Type hints present",
            })

        # check error handling
        has_error_handling = "try:" in code or "except" in code or "HTTPException" in code
        if not has_error_handling:
            quality_issues["missing_error_handling"].append(filename)
            local_results.append({
                "filename":  filename,
                "test_name": "error_handling_check",
                "status":    "warning",
                "detail":    "No error handling found — add try/except or HTTPException",
            })
        else:
            local_results.append({
                "filename":  filename,
                "test_name": "error_handling_check",
                "status":    "pass",
                "detail":    "Error handling present",
            })

        # check async support
        has_async = "async def" in code or "await" in code
        if not has_async:
            quality_issues["missing_async"].append(filename)
            local_results.append({
                "filename":  filename,
                "test_name": "async_check",
                "status":    "warning",
                "detail":    "No async functions — consider async for I/O bound operations",
            })
        else:
            local_results.append({
                "filename":  filename,
                "test_name": "async_check",
                "status":    "pass",
                "detail":    "Async support present",
            })

    criteria         = manifest.get("acceptance_criteria", [])
    code_dump        = " ".join([m.get("code", "") for m in modules])
    criteria_results = []

    keywords_map = {
        "register":       ["register", "signup", "UserCreate"],
        "log in":         ["login", "token", "create_access_token"],
        "dashboard":      ["dashboard", "protected", "Depends"],
        "postgresql":     ["postgresql", "asyncpg", "create_async_engine"],
        "authenticated":  ["Depends", "get_current_user", "verify_token"],
        "persisted":      ["session", "db", "commit", "add"],
    }

    for criterion in criteria:
        matched = False
        evidence = "not found in generated code"
        criterion_lower = criterion.lower()

        for keyword, code_signals in keywords_map.items():
            if keyword in criterion_lower:
                for signal in code_signals:
                    if signal in code_dump:
                        matched   = True
                        evidence  = f"'{signal}' found in generated code"
                        break

        criteria_results.append({
            "criterion": criterion,
            "status":    "met" if matched else "partial",
            "evidence":  evidence,
        })

    # check if critical security findings are still unresolved
    critical_blockers = [
        f"{f['filename']}: {f['rule']}"
        for f in security.get("findings", [])
        if f["severity"] == "critical"
    ]

    return {
        "local_results":      local_results,
        "criteria_results":   criteria_results,
        "quality_issues":     quality_issues,
        "critical_blockers":  critical_blockers,
    }


def quality_agent(state: DevState) -> dict:
    manifest  = state["intent_manifest"]
    arch      = state["architecture"]
    code      = state["generated_code"]
    security  = state["security_report"]
    explain   = state["explainability_docs"]
    modules   = code.get("modules", [])

    # ── local checks first ─────────────────────────────────────────
    local     = run_local_quality_checks(state)

    code_dump = "\n\n".join([
        f"### {m['filename']}\n{m['code']}"
        for m in modules
    ])

    prompt = f"""
You are a senior software quality engineer performing a final quality assessment.

Original Acceptance Criteria:
{json.dumps(manifest.get('acceptance_criteria', []), indent=2)}

Architecture Pattern: {arch.get('selected_pattern')}

Generated Code:
{code_dump}

Local Quality Check Results:
{json.dumps(local['local_results'], indent=2)}

Acceptance Criteria Pre-check:
{json.dumps(local['criteria_results'], indent=2)}

Security Report Summary:
- Overall risk    : {security.get('overall_security_risk')}
- Passed          : {security.get('passed')}
- Critical count  : {sum(1 for f in security.get('findings', []) if f['severity'] == 'critical')}
- Critical blockers: {json.dumps(local['critical_blockers'], indent=2)}

Explainability Coverage:
- Decisions documented : {len(explain.get('decision_log', []))}
- Modules explained    : {len(explain.get('module_explanations', []))}
- Glossary terms       : {len(explain.get('glossary', []))}

Your tasks:
1. Validate every acceptance criterion against the generated code
2. Assess overall code quality — docstrings, type hints, error handling, async
3. Check if security findings are acknowledged and have a remediation path
4. Score the overall quality from 0-100 using this rubric:
   - Acceptance criteria met      : 30 points
   - Code quality                 : 25 points
   - Security findings addressed  : 25 points
   - Explainability coverage      : 20 points
5. Set passed = true ONLY if score >= 70 AND no unresolved critical security findings
6. Give specific, actionable recommendations

Respond ONLY with valid JSON matching this schema, no explanation, no markdown:
{QUALITY_SCHEMA}
"""

    response = llm.invoke(prompt)
    report   = extract_json(response.text)

    # always override security_integration with our local check
    report["security_integration"] = {
        "security_findings_addressed": len(local["critical_blockers"]) == 0,
        "critical_blockers":           local["critical_blockers"],
        "ready_for_deploy":            len(local["critical_blockers"]) == 0
                                       and report.get("overall_quality_score", 0) >= 70,
    }

    # passed only if no critical blockers AND score >= 70
    critical_count = sum(
        1 for f in security.get("findings", [])
        if f["severity"] == "critical"
    )
    report["passed"] = (
        critical_count == 0
        and report.get("overall_quality_score", 0) >= 70
    )

    score   = report.get("overall_quality_score", 0)
    passed  = report.get("passed", False)

    audit_entry = make_audit_entry(
        agent   = "quality_agent",
        summary = (
            f"Quality assessment complete — "
            f"score: {score}/100 | "
            f"passed: {passed} | "
            f"critical blockers: {len(local['critical_blockers'])}"
        ),
        data    = {
            "score":             score,
            "passed":            passed,
            "critical_blockers": local["critical_blockers"],
            "criteria_met":      sum(
                1 for c in report.get("acceptance_criteria_check", [])
                if c["status"] == "met"
            ),
            "recommendations":   report.get("recommendations", []),
        }
    )

    return {
        "quality_report": report,
        "audit_log":      [audit_entry],
    }

In [1]:
def main():
    state: DevState = {
        "raw_input":           "Build a FastAPI web app with login and dashboard using PostgreSQL",
        "intent_manifest":     None,
        "compliance_rules":    None,
        "ip_clearance":        None,
        "architecture":        None,
        "generated_code":      None,
        "explainability_docs": None,
        "security_report":     None,
        "quality_report":      None,
        "hitl_decisions":      [],
        "audit_log":           [],
        "drift_alerts":        None,
    }

    # ── agent 1 ──
    print("=== intent_agent ===")
    result = intent_agent(state)
    state["intent_manifest"] = result["intent_manifest"]
    state["audit_log"] += result["audit_log"]

    # ── agent 2 ──
    print("\n=== ip_guard_agent ===")
    result = ip_guard_agent(state)
    state["ip_clearance"] = result["ip_clearance"]
    state["audit_log"] += result["audit_log"]

    # ── HITL gate 1 ────────────────────────────────────────────────
    max_iterations = 3
    iteration      = 0
    gate1_approved = False

    while not gate1_approved and iteration < max_iterations:
        iteration += 1
        print(f"\n=== hitl_gate_1 (attempt {iteration}) ===")

        result = hitl_gate_1(state)
        state["hitl_decisions"]  = result["hitl_decisions"]
        state["intent_manifest"] = result["intent_manifest"]
        state["raw_input"]       = result["raw_input"]
        state["audit_log"]      += result["audit_log"]

        last = state["hitl_decisions"][-1]

        if last["choice"] == "R":
            print(f"\n>> Rejected — reason: {last['feedback']}")
            print(">> Re-running intent_agent with feedback...")
            result = intent_agent(state)
            state["intent_manifest"] = result["intent_manifest"]
            state["audit_log"] += result["audit_log"]

            print(">> Re-running ip_guard_agent on updated manifest...")
            result = ip_guard_agent(state)
            state["ip_clearance"] = result["ip_clearance"]
            state["audit_log"] += result["audit_log"]

            print("\n>> Updated manifest modules:")
            for m in state["intent_manifest"].get("modules", []):
                print(f"   - {m['name']} → {', '.join(m.get('tech_stack', []))}")

        elif last["choice"] == "M" and last.get("extra_notes"):
            print(f"\n>> Modified — applying: {last['extra_notes']}")
            state["raw_input"] = (
                f"{state['raw_input']}\n\n"
                f"[Human modification]: {last['extra_notes']}\n"
                f"Please update the intent manifest to reflect this change exactly."
            )
            result = intent_agent(state)
            state["intent_manifest"] = result["intent_manifest"]
            state["audit_log"] += result["audit_log"]

            result = ip_guard_agent(state)
            state["ip_clearance"] = result["ip_clearance"]
            state["audit_log"] += result["audit_log"]

            gate1_approved = True

        else:
            gate1_approved = True
            print("\n>> Approved — proceeding to compliance_agent")

    if not gate1_approved:
        print(f"\n>> Max iterations reached — escalating")
        return

    # ── compliance_agent ────────────────────────────────────────────
    print("\n=== compliance_agent ===")
    result = compliance_agent(state)
    state["compliance_rules"] = result["compliance_rules"]
    state["audit_log"] += result["audit_log"]

    print("\n[ Applicable Frameworks ]")
    for fw in state["compliance_rules"].get("applicable_frameworks", []):
        print(f"  {fw['priority'].upper():12s} {fw['name']}")
        print(f"               {fw['reason']}")

    print("\n[ Consolidated Rules ]")
    for i, r in enumerate(state["compliance_rules"].get("consolidated_rules", []), 1):
        print(f"  {i:2}. [{r['framework']}] {r['rule']}")
        print(f"      Hint: {r['implementation_hint']}")

    print("\n[ Gaps Detected ]")
    gaps = state["compliance_rules"].get("gaps", [])
    if gaps:
        for g in gaps:
            print(f"  ! {g}")
    else:
        print("  None detected")

    print(f"\n[ Overall Compliance Risk ] "
          f"{state['compliance_rules'].get('overall_compliance_risk', '').upper()}")

    # ── architecture_agent ──────────────────────────────────────────
    print("\n=== architecture_agent ===")
    result = architecture_agent(state)
    state["architecture"] = result["architecture"]
    state["audit_log"] += result["audit_log"]

    arch = state["architecture"]

    print(f"\n[ Selected Pattern ]  {arch.get('selected_pattern', '').upper()}")
    print(f"  Rationale: {arch.get('pattern_rationale')}")

    print("\n[ Trade-off Matrix ]")
    for k, v in arch.get("trade_off_matrix", {}).items():
        print(f"  {k:20s} {v.upper()}")

    print("\n[ Layers ]")
    for layer in arch.get("layers", []):
        print(f"\n  {layer['name']}")
        print(f"    Responsibility : {layer['responsibility']}")
        print(f"    Tech           : {', '.join(layer.get('tech', []))}")
        print(f"    Components     : {', '.join(layer.get('components', []))}")
        if layer.get("compliance_controls"):
            print(f"    Satisfies      : {', '.join(layer.get('compliance_controls', []))}")

    print("\n[ Infrastructure ]")
    for k, v in arch.get("infrastructure", {}).items():
        print(f"  {k:15s} : {v}")

    print("\n[ Security Controls ]")
    for sc in arch.get("security_controls", []):
        print(f"  + {sc}")

    print("\n[ Compliance Gaps Addressed ]")
    for g in arch.get("gaps_addressed", []):
        print(f"  [resolved] {g}")

    if arch.get("residual_risks"):
        print("\n[ Residual Risks ]")
        for r in arch.get("residual_risks", []):
            print(f"  [!] {r}")

    # ── HITL gate 2 ─────────────────────────────────────────────────
    max_iterations = 3
    iteration      = 0
    gate2_approved = False

    while not gate2_approved and iteration < max_iterations:
        iteration += 1
        print(f"\n=== hitl_gate_2 (attempt {iteration}) ===")

        result = hitl_gate_2(state)
        state["hitl_decisions"] = result["hitl_decisions"]
        state["architecture"]   = result["architecture"]
        state["raw_input"]      = result["raw_input"]
        state["audit_log"]     += result["audit_log"]

        last = state["hitl_decisions"][-1]

        if last["choice"] == "R":
            print(f"\n>> Rejected — reason: {last['feedback']}")
            print(">> Re-running architecture_agent with feedback...")
            result = architecture_agent(state)
            state["architecture"] = result["architecture"]
            state["audit_log"]   += result["audit_log"]

            print("\n>> Updated layers:")
            for layer in state["architecture"].get("layers", []):
                print(f"   - {layer['name']} → {', '.join(layer.get('tech', []))}")

        elif last["choice"] == "M" and last.get("extra_notes"):
            print(f"\n>> Modified — constraint added: {last['extra_notes']}")
            gate2_approved = True

        else:
            gate2_approved = True
            print("\n>> Approved — proceeding to codegen agent")

    if not gate2_approved:
        print(f"\n>> Max iterations reached — escalating")
        return

    # ── codegen_agent ───────────────────────────────────────────────
    print("\n=== codegen_agent ===")
    result = codegen_agent(state)
    state["generated_code"] = result["generated_code"]
    state["audit_log"] += result["audit_log"]

    code = state["generated_code"]

    print("\n[ Project Structure ]")
    for f in code.get("project_structure", []):
        print(f"  {f}")

    print("\n[ Generated Modules ]")
    for m in code.get("modules", []):
        print(f"\n  {m['filename']}  [{m['layer']}]")
        print(f"    {m['description']}")
        print(f"    Rationale  : {m['rationale']}")
        if m.get("compliance_controls"):
            print(f"    Satisfies  : {', '.join(m['compliance_controls'])}")

    print("\n[ Dependencies ]")
    for dep in code.get("dependencies", []):
        print(f"  {dep}")

    print("\n[ Setup Instructions ]")
    for step in code.get("setup_instructions", []):
        print(f"  → {step}")

    print("\n[ Generated Code Preview ]")
    for m in code.get("modules", []):
        print(f"\n{'='*50}")
        print(f"  FILE: {m['filename']}")
        print(f"{'='*50}")
        print(m.get("code", ""))
    
    # ── security_agent ──────────────────────────────────────────────
    print("\n=== security_agent ===")
    result = security_agent(state)
    state["security_report"] = result["security_report"]
    state["audit_log"] += result["audit_log"]

    sec = state["security_report"]

    print(f"\n[ Overall Security Risk ]  {sec.get('overall_security_risk', '').upper()}")
    print(f"[ Passed ]  {sec.get('passed')}")

    print("\n[ Findings ]")
    findings = sec.get("findings", [])
    if findings:
        for f in findings:
            sev = f.get("severity", "").upper()
            print(f"\n  [{sev}] {f['filename']}")
            print(f"    Rule     : {f['rule']}")
            print(f"    OWASP    : {f['owasp_ref']}")
            print(f"    Code     : {f['line_hint']}")
            print(f"    Fix      : {f['fix']}")
    else:
        print("  No findings")

    print("\n[ Compliance Tag Coverage ]")
    cov = sec.get("compliance_tag_coverage", {})
    print(f"  Coverage : {cov.get('coverage_percent')}%")
    if cov.get("files_without_tags"):
        print(f"  Missing tags in: {', '.join(cov['files_without_tags'])}")

    if sec.get("unlicensed_imports"):
        print("\n[ Unlicensed Imports ]")
        for u in sec.get("unlicensed_imports", []):
            print(f"  [!] {u}")

    print(f"\n[ Summary ]\n  {sec.get('summary')}")

    # route based on security outcome
    if not sec.get("passed"):
        print("\n>> Security scan FAILED — critical findings must be fixed")
        print(">> In LangGraph this routes back to codegen_agent")
        print(">> For now printing what would be fixed...")
    else:
        print("\n>> Security scan PASSED — proceeding to quality_agent")

    # ── explainability_agent ────────────────────────────────────────
    print("\n=== explainability_agent ===")
    result = explainability_agent(state)
    state["explainability_docs"] = result["explainability_docs"]
    state["audit_log"] += result["audit_log"]

    docs = state["explainability_docs"]

    print("\n[ Decision Log ]")
    for d in docs.get("decision_log", []):
        print(f"\n  Decision : {d['decision_point']}")
        print(f"  What     : {d['what_was_decided']}")
        print(f"  Why      : {d['why']}")
        if d.get("alternatives_considered"):
            print(f"  Alternatives : {', '.join(d['alternatives_considered'])}")
        if d.get("trade_offs_accepted"):
            print(f"  Trade-offs   : {', '.join(d['trade_offs_accepted'])}")
        if d.get("constraint_satisfied"):
            print(f"  Satisfies    : {', '.join(d['constraint_satisfied'])}")

    print("\n[ Module Explanations ]")
    for m in docs.get("module_explanations", []):
        print(f"\n  {m['filename']}")
        print(f"  Purpose  : {m['purpose']}")
        if m.get("key_decisions"):
            print(f"  Key decisions:")
            for kd in m["key_decisions"]:
                print(f"    - {kd}")
        if m.get("compliance_mapping"):
            print(f"  Compliance:")
            for cm in m["compliance_mapping"]:
                print(f"    - {cm}")

    print("\n[ Glossary ]")
    for g in docs.get("glossary", []):
        print(f"  {g['term']:25s} — {g['plain_english']}")

    print("\n[ Audit Narrative ]")
    print(f"  {docs.get('audit_narrative', '')}")

    # ── quality_agent ───────────────────────────────────────────────
    print("\n=== quality_agent ===")
    result = quality_agent(state)
    state["quality_report"] = result["quality_report"]
    state["audit_log"] += result["audit_log"]

    q = state["quality_report"]

    print(f"\n[ Overall Quality Score ]  {q.get('overall_quality_score')}/100")
    print(f"[ Passed ]  {q.get('passed')}")

    print("\n[ Acceptance Criteria ]")
    for c in q.get("acceptance_criteria_check", []):
        status_tag = {"met": "[ok]", "partial": "[~]", "not_met": "[!!]"}.get(
            c["status"], "[?]"
        )
        print(f"  {status_tag:6s} {c['criterion']}")
        print(f"         {c['evidence']}")

    print("\n[ Code Quality ]")
    cq = q.get("code_quality", {})
    print(f"  Docstrings     : {'yes' if cq.get('has_docstrings') else 'NO'}")
    print(f"  Type hints     : {'yes' if cq.get('has_type_hints') else 'NO'}")
    print(f"  Error handling : {'yes' if cq.get('has_error_handling') else 'NO'}")
    print(f"  Async support  : {'yes' if cq.get('has_async_support') else 'NO'}")
    if cq.get("missing_docstrings_in"):
        print(f"  Missing docstrings  : {', '.join(cq['missing_docstrings_in'])}")
    if cq.get("missing_error_handling_in"):
        print(f"  Missing error handling : {', '.join(cq['missing_error_handling_in'])}")

    print("\n[ Security Integration ]")
    si = q.get("security_integration", {})
    print(f"  Findings addressed : {si.get('security_findings_addressed')}")
    print(f"  Ready for deploy   : {si.get('ready_for_deploy')}")
    if si.get("critical_blockers"):
        print(f"  Critical blockers  :")
        for cb in si["critical_blockers"]:
            print(f"    [!!] {cb}")

    print("\n[ Recommendations ]")
    for r in q.get("recommendations", []):
        print(f"  → {r}")

    print(f"\n[ Summary ]\n  {q.get('summary')}")

    if not q.get("passed"):
        print("\n>> Quality check FAILED — routing back to codegen_agent in LangGraph")
    else:
        print("\n>> Quality check PASSED — proceeding to audit_agent")

    # ── audit_agent ─────────────────────────────────────────────────
    print("\n=== audit_agent ===")
    result = audit_agent(state)
    state["audit_log"]     += result["audit_log"]
    state["quality_report"] = result["quality_report"]

    audit = state["quality_report"]["final_audit"]

    print(f"\n[ Final Status ]  "
          f"{audit.get('final_status', '').upper().replace('_', ' ')}")

    print("\n[ Pipeline Summary ]")
    ps = audit.get("pipeline_summary", {})
    print(f"  Agents run        : {ps.get('total_agents_run')}")
    print(f"  HITL decisions    : {ps.get('total_hitl_decisions')}")
    print(f"  Security findings : {ps.get('total_findings')}")
    print(f"  Pipeline passed   : {ps.get('pipeline_passed')}")

    if ps.get("blocking_issues"):
        print("\n[ Blocking Issues ]")
        for b in ps["blocking_issues"]:
            print(f"  [!!] {b}")

    print("\n[ Compliance Sign-off ]")
    cs = audit.get("compliance_sign_off", {})
    print(f"  GDPR verified     : {cs.get('gdpr_controls_verified')}")
    print(f"  OWASP verified    : {cs.get('owasp_controls_verified')}")
    print(f"  IP clearance      : {cs.get('ip_clearance_verified')}")
    print(f"  Gaps resolved     : {cs.get('gaps_resolved')}")
    if cs.get("unresolved_items"):
        for u in cs["unresolved_items"]:
            print(f"  [!] Unresolved: {u}")

    print("\n[ Human Accountability Trail ]")
    for h in audit.get("human_accountability", []):
        print(f"  {h['gate']:15s} | {h['approver']:10s} | "
              f"{h['decision']} | ack risks: {h['risks_acknowledged']} "
              f"| {h['timestamp']}")

    print("\n[ Immutable Digest ]")
    d = audit.get("immutable_digest", {})
    for k, v in d.items():
        print(f"  {k:20s} : {v}")

    print(f"\n[ Audit Sign-off Note ]\n  {audit.get('sign_off_note')}")

    print("\n=== Full Audit Log ===")
    for entry in state["audit_log"]:
        print(f"  [{entry['timestamp']}] {entry['agent']:20s} — {entry['summary']}")


if __name__ == "__main__":
    main()

=== intent_agent ===


NameError: name 'intent_agent' is not defined

In [20]:
AUDIT_SCHEMA = """
{
  "pipeline_summary": {
    "total_agents_run":      "number",
    "total_hitl_decisions":  "number",
    "total_findings":        "number",
    "pipeline_passed":       "boolean",
    "blocking_issues":       ["string"]
  },
  "compliance_sign_off": {
    "gdpr_controls_verified":  "boolean",
    "owasp_controls_verified": "boolean",
    "ip_clearance_verified":   "boolean",
    "gaps_resolved":           "boolean",
    "unresolved_items":        ["string"]
  },
  "human_accountability": [
    {
      "gate":      "string",
      "approver":  "string",
      "decision":  "string",
      "timestamp": "string",
      "risks_acknowledged": "boolean",
      "notes":     "string"
    }
  ],
  "immutable_digest": {
    "intent_hash":       "string — md5 of intent manifest",
    "architecture_hash": "string — md5 of architecture",
    "code_hash":         "string — md5 of generated code filenames",
    "audit_chain_hash":  "string — md5 of full audit log"
  },
  "final_status":  "approved_for_deploy | blocked | requires_remediation",
  "sign_off_note": "string — one paragraph final audit statement"
}
"""

def compute_hash(data) -> str:
    """Compute MD5 hash of any serializable data for audit digest."""
    raw = json.dumps(data, sort_keys=True, default=str)
    return hashlib.md5(raw.encode()).hexdigest()


def audit_agent(state: DevState) -> dict:
    manifest    = state["intent_manifest"]
    arch        = state["architecture"]
    compliance  = state["compliance_rules"]
    ip          = state["ip_clearance"]
    code        = state["generated_code"]
    security    = state["security_report"]
    quality     = state["quality_report"]
    explain     = state["explainability_docs"]
    decisions   = state["hitl_decisions"]
    audit_log   = state["audit_log"]

    # ── compute immutable digest ────────────────────────────────────
    immutable_digest = {
        "intent_hash":       compute_hash(manifest),
        "architecture_hash": compute_hash(arch),
        "code_hash":         compute_hash([
            m["filename"] for m in code.get("modules", [])
        ]),
        "audit_chain_hash":  compute_hash(audit_log),
    }

    # ── build human accountability trail ───────────────────────────
    accountability = []
    for d in decisions:
        accountability.append({
            "gate":               d.get("gate"),
            "approver":           d.get("approver"),
            "decision":           d.get("choice"),
            "timestamp":          d.get("timestamp"),
            "risks_acknowledged": d.get("risk_acknowledged", False),
            "notes":              d.get("extra_notes") or d.get("feedback") or "none",
        })

    # ── check blocking issues ───────────────────────────────────────
    blocking_issues = []

    # critical security findings
    critical_findings = [
        f"{f['filename']}: {f['rule']}"
        for f in security.get("findings", [])
        if f["severity"] == "critical"
    ]
    blocking_issues.extend(critical_findings)

    # quality not passed
    if not quality.get("passed"):
        blocking_issues.append(
            f"Quality score {quality.get('overall_quality_score')}/100 — below threshold"
        )

    # unmet acceptance criteria
    unmet = [
        c["criterion"]
        for c in quality.get("acceptance_criteria_check", [])
        if c["status"] == "not_met"
    ]
    blocking_issues.extend([f"Unmet criterion: {u}" for u in unmet])

    # ── compliance verification ─────────────────────────────────────
    frameworks    = [
        f["name"] for f in compliance.get("applicable_frameworks", [])
    ]
    gaps          = compliance.get("gaps", [])
    gaps_addressed = arch.get("gaps_addressed", [])
    unresolved    = [
        g for g in gaps
        if not any(
            any(w in a.lower() for w in g.lower().split()[:3])
            for a in gaps_addressed
        )
    ]

    compliance_sign_off = {
        "gdpr_controls_verified":  "GDPR" in frameworks,
        "owasp_controls_verified": "OWASP Top 10" in frameworks,
        "ip_clearance_verified":   ip.get("overall_risk") in ("low", "medium"),
        "gaps_resolved":           len(unresolved) == 0,
        "unresolved_items":        unresolved,
    }

    # ── determine final status ──────────────────────────────────────
    if blocking_issues:
        final_status = "requires_remediation"
    elif not quality.get("passed"):
        final_status = "blocked"
    else:
        final_status = "approved_for_deploy"

    # ── LLM generates the sign-off note ────────────────────────────
    prompt = f"""
You are a senior compliance auditor writing a final audit statement
for a software system built by an AI pipeline.

Pipeline Summary:
- Agents run        : {len(audit_log)}
- HITL decisions    : {len(decisions)}
- Security findings : {len(security.get('findings', []))}
- Quality score     : {quality.get('overall_quality_score')}/100
- Final status      : {final_status}

Human Accountability Trail:
{json.dumps(accountability, indent=2)}

Blocking Issues:
{json.dumps(blocking_issues, indent=2) if blocking_issues else "None"}

Compliance Sign-off:
{json.dumps(compliance_sign_off, indent=2)}

Immutable Digest:
{json.dumps(immutable_digest, indent=2)}

Write a single paragraph audit statement that:
1. States what was built and by whom it was reviewed
2. Confirms which compliance frameworks were verified
3. Lists any blocking issues that prevent deployment
4. States the final deployment status clearly
5. Is written for a regulator or external auditor

Return ONLY the paragraph as a plain string — no JSON, no markdown.
"""

    response  = llm.invoke(prompt)
    sign_off  = response.text.strip()

    # ── build final report ──────────────────────────────────────────
    report = {
        "pipeline_summary": {
            "total_agents_run":     len(audit_log),
            "total_hitl_decisions": len(decisions),
            "total_findings":       len(security.get("findings", [])),
            "pipeline_passed":      len(blocking_issues) == 0,
            "blocking_issues":      blocking_issues,
        },
        "compliance_sign_off":  compliance_sign_off,
        "human_accountability": accountability,
        "immutable_digest":     immutable_digest,
        "final_status":         final_status,
        "sign_off_note":        sign_off,
    }

    status_display = {
        "approved_for_deploy":  "APPROVED FOR DEPLOY",
        "blocked":              "BLOCKED",
        "requires_remediation": "REQUIRES REMEDIATION",
    }.get(final_status, final_status.upper())

    audit_entry = make_audit_entry(
        agent   = "audit_agent",
        summary = (
            f"Final audit complete — "
            f"status: {status_display} | "
            f"blocking issues: {len(blocking_issues)} | "
            f"digest: {immutable_digest['audit_chain_hash'][:8]}..."
        ),
        data    = {
            "final_status":         final_status,
            "blocking_issues":      len(blocking_issues),
            "pipeline_passed":      len(blocking_issues) == 0,
            "immutable_digest":     immutable_digest,
            "compliance_sign_off":  compliance_sign_off,
        }
    )

    return {
        "audit_log": [audit_entry],
        "quality_report": {
            **quality,
            "final_audit": report,
        }
    }

In [ ]:
def main():
    state: DevState = {
        "raw_input":           "Build a FastAPI web app with login and dashboard using PostgreSQL",
        "intent_manifest":     None,
        "compliance_rules":    None,
        "ip_clearance":        None,
        "architecture":        None,
        "generated_code":      None,
        "explainability_docs": None,
        "security_report":     None,
        "quality_report":      None,
        "hitl_decisions":      [],
        "audit_log":           [],
        "drift_alerts":        None,
    }

    # ── agent 1 ──
    print("=== intent_agent ===")
    result = intent_agent(state)
    state["intent_manifest"] = result["intent_manifest"]
    state["audit_log"] += result["audit_log"]

    # ── agent 2 ──
    print("\n=== ip_guard_agent ===")
    result = ip_guard_agent(state)
    state["ip_clearance"] = result["ip_clearance"]
    state["audit_log"] += result["audit_log"]

    # ── HITL gate 1 ────────────────────────────────────────────────
    max_iterations = 3
    iteration      = 0
    gate1_approved = False

    while not gate1_approved and iteration < max_iterations:
        iteration += 1
        print(f"\n=== hitl_gate_1 (attempt {iteration}) ===")

        result = hitl_gate_1(state)
        state["hitl_decisions"]  = result["hitl_decisions"]
        state["intent_manifest"] = result["intent_manifest"]
        state["raw_input"]       = result["raw_input"]
        state["audit_log"]      += result["audit_log"]

        last = state["hitl_decisions"][-1]

        if last["choice"] == "R":
            print(f"\n>> Rejected — reason: {last['feedback']}")
            print(">> Re-running intent_agent with feedback...")
            result = intent_agent(state)
            state["intent_manifest"] = result["intent_manifest"]
            state["audit_log"] += result["audit_log"]

            print(">> Re-running ip_guard_agent on updated manifest...")
            result = ip_guard_agent(state)
            state["ip_clearance"] = result["ip_clearance"]
            state["audit_log"] += result["audit_log"]

            print("\n>> Updated manifest modules:")
            for m in state["intent_manifest"].get("modules", []):
                print(f"   - {m['name']} → {', '.join(m.get('tech_stack', []))}")

        elif last["choice"] == "M" and last.get("extra_notes"):
            print(f"\n>> Modified — applying: {last['extra_notes']}")
            state["raw_input"] = (
                f"{state['raw_input']}\n\n"
                f"[Human modification]: {last['extra_notes']}\n"
                f"Please update the intent manifest to reflect this change exactly."
            )
            result = intent_agent(state)
            state["intent_manifest"] = result["intent_manifest"]
            state["audit_log"] += result["audit_log"]

            result = ip_guard_agent(state)
            state["ip_clearance"] = result["ip_clearance"]
            state["audit_log"] += result["audit_log"]

            gate1_approved = True

        else:
            gate1_approved = True
            print("\n>> Approved — proceeding to compliance_agent")

    if not gate1_approved:
        print(f"\n>> Max iterations reached — escalating")
        return

    # ── compliance_agent ────────────────────────────────────────────
    print("\n=== compliance_agent ===")
    result = compliance_agent(state)
    state["compliance_rules"] = result["compliance_rules"]
    state["audit_log"] += result["audit_log"]

    print("\n[ Applicable Frameworks ]")
    for fw in state["compliance_rules"].get("applicable_frameworks", []):
        print(f"  {fw['priority'].upper():12s} {fw['name']}")
        print(f"               {fw['reason']}")

    print("\n[ Consolidated Rules ]")
    for i, r in enumerate(state["compliance_rules"].get("consolidated_rules", []), 1):
        print(f"  {i:2}. [{r['framework']}] {r['rule']}")
        print(f"      Hint: {r['implementation_hint']}")

    print("\n[ Gaps Detected ]")
    gaps = state["compliance_rules"].get("gaps", [])
    if gaps:
        for g in gaps:
            print(f"  ! {g}")
    else:
        print("  None detected")

    print(f"\n[ Overall Compliance Risk ] "
          f"{state['compliance_rules'].get('overall_compliance_risk', '').upper()}")

    # ── architecture_agent ──────────────────────────────────────────
    print("\n=== architecture_agent ===")
    result = architecture_agent(state)
    state["architecture"] = result["architecture"]
    state["audit_log"] += result["audit_log"]

    arch = state["architecture"]

    print(f"\n[ Selected Pattern ]  {arch.get('selected_pattern', '').upper()}")
    print(f"  Rationale: {arch.get('pattern_rationale')}")

    print("\n[ Trade-off Matrix ]")
    for k, v in arch.get("trade_off_matrix", {}).items():
        print(f"  {k:20s} {v.upper()}")

    print("\n[ Layers ]")
    for layer in arch.get("layers", []):
        print(f"\n  {layer['name']}")
        print(f"    Responsibility : {layer['responsibility']}")
        print(f"    Tech           : {', '.join(layer.get('tech', []))}")
        print(f"    Components     : {', '.join(layer.get('components', []))}")
        if layer.get("compliance_controls"):
            print(f"    Satisfies      : {', '.join(layer.get('compliance_controls', []))}")

    print("\n[ Infrastructure ]")
    for k, v in arch.get("infrastructure", {}).items():
        print(f"  {k:15s} : {v}")

    print("\n[ Security Controls ]")
    for sc in arch.get("security_controls", []):
        print(f"  + {sc}")

    print("\n[ Compliance Gaps Addressed ]")
    for g in arch.get("gaps_addressed", []):
        print(f"  [resolved] {g}")

    if arch.get("residual_risks"):
        print("\n[ Residual Risks ]")
        for r in arch.get("residual_risks", []):
            print(f"  [!] {r}")

    # ── HITL gate 2 ─────────────────────────────────────────────────
    max_iterations = 3
    iteration      = 0
    gate2_approved = False

    while not gate2_approved and iteration < max_iterations:
        iteration += 1
        print(f"\n=== hitl_gate_2 (attempt {iteration}) ===")

        result = hitl_gate_2(state)
        state["hitl_decisions"] = result["hitl_decisions"]
        state["architecture"]   = result["architecture"]
        state["raw_input"]      = result["raw_input"]
        state["audit_log"]     += result["audit_log"]

        last = state["hitl_decisions"][-1]

        if last["choice"] == "R":
            print(f"\n>> Rejected — reason: {last['feedback']}")
            print(">> Re-running architecture_agent with feedback...")
            result = architecture_agent(state)
            state["architecture"] = result["architecture"]
            state["audit_log"]   += result["audit_log"]

            print("\n>> Updated layers:")
            for layer in state["architecture"].get("layers", []):
                print(f"   - {layer['name']} → {', '.join(layer.get('tech', []))}")

        elif last["choice"] == "M" and last.get("extra_notes"):
            print(f"\n>> Modified — constraint added: {last['extra_notes']}")
            gate2_approved = True

        else:
            gate2_approved = True
            print("\n>> Approved — proceeding to codegen agent")

    if not gate2_approved:
        print(f"\n>> Max iterations reached — escalating")
        return

    # ── codegen_agent ───────────────────────────────────────────────
    print("\n=== codegen_agent ===")
    result = codegen_agent(state)
    state["generated_code"] = result["generated_code"]
    state["audit_log"] += result["audit_log"]

    code = state["generated_code"]

    print("\n[ Project Structure ]")
    for f in code.get("project_structure", []):
        print(f"  {f}")

    print("\n[ Generated Modules ]")
    for m in code.get("modules", []):
        print(f"\n  {m['filename']}  [{m['layer']}]")
        print(f"    {m['description']}")
        print(f"    Rationale  : {m['rationale']}")
        if m.get("compliance_controls"):
            print(f"    Satisfies  : {', '.join(m['compliance_controls'])}")

    print("\n[ Dependencies ]")
    for dep in code.get("dependencies", []):
        print(f"  {dep}")

    print("\n[ Setup Instructions ]")
    for step in code.get("setup_instructions", []):
        print(f"  → {step}")

    print("\n[ Generated Code Preview ]")
    for m in code.get("modules", []):
        print(f"\n{'='*50}")
        print(f"  FILE: {m['filename']}")
        print(f"{'='*50}")
        print(m.get("code", ""))
    
    # ── security_agent ──────────────────────────────────────────────
    print("\n=== security_agent ===")
    result = security_agent(state)
    state["security_report"] = result["security_report"]
    state["audit_log"] += result["audit_log"]

    sec = state["security_report"]

    print(f"\n[ Overall Security Risk ]  {sec.get('overall_security_risk', '').upper()}")
    print(f"[ Passed ]  {sec.get('passed')}")

    print("\n[ Findings ]")
    findings = sec.get("findings", [])
    if findings:
        for f in findings:
            sev = f.get("severity", "").upper()
            print(f"\n  [{sev}] {f['filename']}")
            print(f"    Rule     : {f['rule']}")
            print(f"    OWASP    : {f['owasp_ref']}")
            print(f"    Code     : {f['line_hint']}")
            print(f"    Fix      : {f['fix']}")
    else:
        print("  No findings")

    print("\n[ Compliance Tag Coverage ]")
    cov = sec.get("compliance_tag_coverage", {})
    print(f"  Coverage : {cov.get('coverage_percent')}%")
    if cov.get("files_without_tags"):
        print(f"  Missing tags in: {', '.join(cov['files_without_tags'])}")

    if sec.get("unlicensed_imports"):
        print("\n[ Unlicensed Imports ]")
        for u in sec.get("unlicensed_imports", []):
            print(f"  [!] {u}")

    print(f"\n[ Summary ]\n  {sec.get('summary')}")

    # route based on security outcome
    if not sec.get("passed"):
        print("\n>> Security scan FAILED — critical findings must be fixed")
        print(">> In LangGraph this routes back to codegen_agent")
        print(">> For now printing what would be fixed...")
    else:
        print("\n>> Security scan PASSED — proceeding to quality_agent")

    # ── explainability_agent ────────────────────────────────────────
    print("\n=== explainability_agent ===")
    result = explainability_agent(state)
    state["explainability_docs"] = result["explainability_docs"]
    state["audit_log"] += result["audit_log"]

    docs = state["explainability_docs"]

    print("\n[ Decision Log ]")
    for d in docs.get("decision_log", []):
        print(f"\n  Decision : {d['decision_point']}")
        print(f"  What     : {d['what_was_decided']}")
        print(f"  Why      : {d['why']}")
        if d.get("alternatives_considered"):
            print(f"  Alternatives : {', '.join(d['alternatives_considered'])}")
        if d.get("trade_offs_accepted"):
            print(f"  Trade-offs   : {', '.join(d['trade_offs_accepted'])}")
        if d.get("constraint_satisfied"):
            print(f"  Satisfies    : {', '.join(d['constraint_satisfied'])}")

    print("\n[ Module Explanations ]")
    for m in docs.get("module_explanations", []):
        print(f"\n  {m['filename']}")
        print(f"  Purpose  : {m['purpose']}")
        if m.get("key_decisions"):
            print(f"  Key decisions:")
            for kd in m["key_decisions"]:
                print(f"    - {kd}")
        if m.get("compliance_mapping"):
            print(f"  Compliance:")
            for cm in m["compliance_mapping"]:
                print(f"    - {cm}")

    print("\n[ Glossary ]")
    for g in docs.get("glossary", []):
        print(f"  {g['term']:25s} — {g['plain_english']}")

    print("\n[ Audit Narrative ]")
    print(f"  {docs.get('audit_narrative', '')}")

    # ── quality_agent ───────────────────────────────────────────────
    print("\n=== quality_agent ===")
    result = quality_agent(state)
    state["quality_report"] = result["quality_report"]
    state["audit_log"] += result["audit_log"]

    q = state["quality_report"]

    print(f"\n[ Overall Quality Score ]  {q.get('overall_quality_score')}/100")
    print(f"[ Passed ]  {q.get('passed')}")

    print("\n[ Acceptance Criteria ]")
    for c in q.get("acceptance_criteria_check", []):
        status_tag = {"met": "[ok]", "partial": "[~]", "not_met": "[!!]"}.get(
            c["status"], "[?]"
        )
        print(f"  {status_tag:6s} {c['criterion']}")
        print(f"         {c['evidence']}")

    print("\n[ Code Quality ]")
    cq = q.get("code_quality", {})
    print(f"  Docstrings     : {'yes' if cq.get('has_docstrings') else 'NO'}")
    print(f"  Type hints     : {'yes' if cq.get('has_type_hints') else 'NO'}")
    print(f"  Error handling : {'yes' if cq.get('has_error_handling') else 'NO'}")
    print(f"  Async support  : {'yes' if cq.get('has_async_support') else 'NO'}")
    if cq.get("missing_docstrings_in"):
        print(f"  Missing docstrings  : {', '.join(cq['missing_docstrings_in'])}")
    if cq.get("missing_error_handling_in"):
        print(f"  Missing error handling : {', '.join(cq['missing_error_handling_in'])}")

    print("\n[ Security Integration ]")
    si = q.get("security_integration", {})
    print(f"  Findings addressed : {si.get('security_findings_addressed')}")
    print(f"  Ready for deploy   : {si.get('ready_for_deploy')}")
    if si.get("critical_blockers"):
        print(f"  Critical blockers  :")
        for cb in si["critical_blockers"]:
            print(f"    [!!] {cb}")

    print("\n[ Recommendations ]")
    for r in q.get("recommendations", []):
        print(f"  → {r}")

    print(f"\n[ Summary ]\n  {q.get('summary')}")

    if not q.get("passed"):
        print("\n>> Quality check FAILED — routing back to codegen_agent in LangGraph")
    else:
        print("\n>> Quality check PASSED — proceeding to audit_agent")

    # ── audit_agent ─────────────────────────────────────────────────
    print("\n=== audit_agent ===")
    result = audit_agent(state)
    state["audit_log"]     += result["audit_log"]
    state["quality_report"] = result["quality_report"]

    audit = state["quality_report"]["final_audit"]

    print(f"\n[ Final Status ]  "
          f"{audit.get('final_status', '').upper().replace('_', ' ')}")

    print("\n[ Pipeline Summary ]")
    ps = audit.get("pipeline_summary", {})
    print(f"  Agents run        : {ps.get('total_agents_run')}")
    print(f"  HITL decisions    : {ps.get('total_hitl_decisions')}")
    print(f"  Security findings : {ps.get('total_findings')}")
    print(f"  Pipeline passed   : {ps.get('pipeline_passed')}")

    if ps.get("blocking_issues"):
        print("\n[ Blocking Issues ]")
        for b in ps["blocking_issues"]:
            print(f"  [!!] {b}")

    print("\n[ Compliance Sign-off ]")
    cs = audit.get("compliance_sign_off", {})
    print(f"  GDPR verified     : {cs.get('gdpr_controls_verified')}")
    print(f"  OWASP verified    : {cs.get('owasp_controls_verified')}")
    print(f"  IP clearance      : {cs.get('ip_clearance_verified')}")
    print(f"  Gaps resolved     : {cs.get('gaps_resolved')}")
    if cs.get("unresolved_items"):
        for u in cs["unresolved_items"]:
            print(f"  [!] Unresolved: {u}")

    print("\n[ Human Accountability Trail ]")
    for h in audit.get("human_accountability", []):
        print(f"  {h['gate']:15s} | {h['approver']:10s} | "
              f"{h['decision']} | ack risks: {h['risks_acknowledged']} "
              f"| {h['timestamp']}")

    print("\n[ Immutable Digest ]")
    d = audit.get("immutable_digest", {})
    for k, v in d.items():
        print(f"  {k:20s} : {v}")

    print(f"\n[ Audit Sign-off Note ]\n  {audit.get('sign_off_note')}")

    print("\n=== Full Audit Log ===")
    for entry in state["audit_log"]:
        print(f"  [{entry['timestamp']}] {entry['agent']:20s} — {entry['summary']}")


if __name__ == "__main__":
    main()

=== intent_agent ===

=== ip_guard_agent ===

=== hitl_gate_1 (attempt 1) ===

  HITL GATE 1 — INTENT + IP REVIEW

[ Intent Manifest ]
  App type : web app
  Modules  :
    - Authentication Service → FastAPI, Passlib, PyJWT
    - Dashboard Module → FastAPI, Jinja2, HTML/CSS
    - Data Persistence Layer → PostgreSQL, SQLAlchemy, Alembic
  Security constraints  : ['Password hashing with Argon2 or BCrypt', 'JWT token expiration', 'CORS policy configuration']
  Compliance constraints: ['None specified']
  Acceptance criteria   :
    - User can register a new account
    - User can log in and receive an authentication token
    - Dashboard is inaccessible to unauthenticated users
    - Data is successfully persisted in PostgreSQL

[ IP Clearance Report ]
  Overall risk : LOW
  Flagged items: none
  Recommendation: The project stack is composed entirely of permissive licenses; no further legal action is required for commercial deployment.

[ Scanned Libraries ]
  [ok]     FastAPI            